In [1]:
import sqlite3
from datetime import datetime, timedelta
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd

# ========== 1. DATABASE ==========
DB_NAME = "tasks.db"

def init_db():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute('''CREATE TABLE IF NOT EXISTS tasks (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        estimated_minutes INTEGER,
        deadline TEXT,
        priority TEXT,
        category TEXT,
        status TEXT DEFAULT 'pending',
        created_at TEXT DEFAULT CURRENT_TIMESTAMP,
        completed_at TEXT
    )''')
    conn.commit()
    conn.close()

init_db()
print("✅ Database siap")

✅ Database siap


In [2]:
import sqlite3
from datetime import datetime, timedelta
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd

DB_NAME = "tasks.db"

def init_db():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    # Buat tabel jika belum ada
    c.execute('''CREATE TABLE IF NOT EXISTS tasks (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        estimated_minutes INTEGER,
        deadline TEXT,
        priority TEXT,
        category TEXT,
        status TEXT DEFAULT 'pending',
        created_at TEXT DEFAULT CURRENT_TIMESTAMP,
        completed_at TEXT
    )''')
    # Cek apakah kolom completed_at ada, jika tidak tambahkan
    c.execute("PRAGMA table_info(tasks)")
    columns = [col[1] for col in c.fetchall()]
    if 'completed_at' not in columns:
        c.execute("ALTER TABLE tasks ADD COLUMN completed_at TEXT")
    conn.commit()
    conn.close()

init_db()

def add_task(name, minutes, deadline, priority, category):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute('''INSERT INTO tasks (name, estimated_minutes, deadline, priority, category, status)
                 VALUES (?,?,?,?,?, 'pending')''', (name, minutes, deadline, priority, category))
    conn.commit()
    conn.close()

def get_all_tasks_df():
    conn = sqlite3.connect(DB_NAME)
    df = pd.read_sql_query("SELECT id, name, estimated_minutes, deadline, priority, category, status FROM tasks", conn)
    conn.close()
    return df

def clear_all_tasks():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("DELETE FROM tasks")
    conn.commit()
    conn.close()

def complete_task_by_id(task_id):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    # Cek status
    c.execute("SELECT status FROM tasks WHERE id=?", (task_id,))
    row = c.fetchone()
    if not row:
        conn.close()
        return False, "ID tidak ditemukan"
    if row[0] != 'pending':
        conn.close()
        return False, "Tugas wis rampung"
    # Update
    now = datetime.now().isoformat()
    c.execute("UPDATE tasks SET status='completed', completed_at=? WHERE id=? AND status='pending'",
              (now, task_id))
    affected = c.rowcount
    conn.commit()
    conn.close()
    return affected > 0, "Berhasil" if affected else "Gagal update"

# Data contoh jika kosong
def insert_sample():
    df = get_all_tasks_df()
    if df.empty:
        sample = [
            ("Belajar UAP", 100, "2026-06-12", "High", "Akademik"),
            ("Pengerjaan tugas penulisan ilmiah", 60, "2026-06-10", "High", "Akademik"),
            ("Jogging", 35, "2026-06-11", "Medium", "Pribadi"),
            ("Tugas video resume FPO", 90, "2026-06-13", "Medium", "Non akademik"),
            ("Memahami isi notulensi", 30, "2026-06-09", "Medium", "Organisasi")
        ]
        for n,m,d,p,k in sample:
            add_task(n,m,d,p,k)
        print("Contoh tugas ditambahkan.")
insert_sample()

def tampilkan_tabel():
    df = get_all_tasks_df()
    if df.empty:
        return HTML("<p style='color:white'> Tidak ada tugas.</p>")
    df_disp = df.copy()
    df_disp['status'] = df_disp['status'].apply(lambda x: '⏳ Pending' if x=='pending' else '✅ Completed')
    def warna(row):
        if row['priority'] == 'High':
            return 'background-color: #8B0000; color: white'
        elif row['priority'] == 'Medium':
            return 'background-color: #B8860B; color: white'
        else:
            return 'background-color: #006400; color: white'
    styled = df_disp.style.apply(lambda row: [warna(row) for _ in row], axis=1)
    styled = styled.set_table_styles([
        {'selector': 'table', 'props': [('background-color', '#1e1e1e'), ('color', 'white'),
                                        ('border-collapse', 'collapse'), ('width', '100%')]},
        {'selector': 'th', 'props': [('background-color', '#0d47a1'), ('color', '#ffd966'),
                                     ('padding', '10px')]},
        {'selector': 'td', 'props': [('border', '1px solid #555'), ('padding', '8px')]}
    ])
    return styled

def tampil_progress_html():
    df = get_all_tasks_df()
    if df.empty:
        return HTML("<p>Belum ada tugas.</p>")
    total = len(df)
    selesai = len(df[df['status']=='completed'])
    pending = total - selesai
    percent = (selesai / total) * 100 if total > 0 else 0

    html = f"""
    <div style="border:1px solid #555; border-radius:10px; padding:2px; background:#2d2d2d;">
        <div style="background-color:#4CAF50; width:{percent}%; border-radius:8px; text-align:center; color:white; transition:width 0.3s;">
            {percent:.1f}%
        </div>
    </div>
    <p>✅ Selesai: {selesai} | ⏳ Pending: {pending} | Total: {total}</p>
    """
    # Animasi saat 100%
    if percent == 100:
        html += """
        <div id="happy-emoji" style="text-align:center; font-size:40px; animation: bounce 0.5s infinite alternate;">
            😊😁🎉🥳✨
        </div>
        <div style="text-align:center; animation: blink 0.8s infinite; font-size:20px; margin-top:10px;">
            🎉🎉 SELAMAT! SEMUA TUGAS SELESAI! 🎉🎉
        </div>
        <style>
            @keyframes blink {
                0% { color: #ffd966; transform: scale(1); }
                50% { color: #ff6666; transform: scale(1.1); }
                100% { color: #ffd966; transform: scale(1); }
            }
            @keyframes bounce {
                from { transform: translateY(0px); }
                to { transform: translateY(-20px); }
            }
        </style>
        <script>
        (function() {
            if (typeof canvasConfetti === 'undefined') {
                var script = document.createElement('script');
                script.src = 'https://cdn.jsdelivr.net/npm/canvas-confetti@1';
                script.onload = function() {
                    canvasConfetti({particleCount: 300, spread: 100, origin: {y: 0.6}});
                    canvasConfetti({particleCount: 100, spread: 70, origin: {y: 0.4, x: 0.3}});
                    canvasConfetti({particleCount: 100, spread: 70, origin: {y: 0.4, x: 0.7}});
                };
                document.head.appendChild(script);
            } else {
                canvasConfetti({particleCount: 300, spread: 100, origin: {y: 0.6}});
                canvasConfetti({particleCount: 100, spread: 70, origin: {y: 0.4, x: 0.3}});
                canvasConfetti({particleCount: 100, spread: 70, origin: {y: 0.4, x: 0.7}});
            }
        })();
        </script>
        """
    return HTML(html)

def buat_jadwal():
    df = get_all_tasks_df()
    pending = df[df['status']=='pending']
    if pending.empty:
        return HTML("<p> Semua tugas selesai! Tidak ada jadwal.</p>")
    total_menit = 8*60
    used = 0
    mulai = datetime.strptime("09:00","%H:%M")
    html = "<h3 style='color:#ffd966'> Jadwal Hari Ini (8 jam, mulai 09:00)</h3><ul style='list-style:none'>"
    for _,row in pending.iterrows():
        dur = row['estimated_minutes']
        if used + dur <= total_menit:
            selesai = mulai + timedelta(minutes=dur)
            warna = "#9b2c2c" if row['priority']=='High' else "#b87c1c" if row['priority']=='Medium' else "#2c6e2c"
            html += f"<li style='background:{warna}; color:white; margin:8px; padding:10px; border-radius:10px'>"
            html += f"<b>{mulai.strftime('%H:%M')} - {selesai.strftime('%H:%M')}</b> | {row['name']} ({dur} menit) | {row['priority']}</li>"
            mulai = selesai
            used += dur
        else:
            html += f"<li style='background:#444; color:#ccc; margin:8px; padding:10px; border-radius:10px'>⚠️ {row['name']} tidak muat → besok</li>"
    html += f"</ul><p>⏱ Terpakai: {used/60:.1f} jam</p>"
    return HTML(html)

# Widget
nama = widgets.Text(description="📝 Nama:", layout=widgets.Layout(width='300px'))
waktu = widgets.IntSlider(min=5,max=300,step=5,value=30,description="⏱ Menit:")
prioritas = widgets.Dropdown(options=['High','Medium','Low'],value='Medium',description="⚡ Prioritas:")
deadline = widgets.DatePicker(description="📅 Deadline:")
kategori = widgets.Text(description="📂 Kategori:")

btn_tambah = widgets.Button(description="➕ Tambah Tugas", button_style='success')
btn_reset = widgets.Button(description="🗑 Hapus Semua", button_style='danger')

out_tabel = widgets.Output()
out_progress = widgets.Output()
out_jadwal = widgets.Output()
out_status = widgets.Output()

def refresh():
    with out_tabel: clear_output(); display(tampilkan_tabel())
    with out_progress: clear_output(); display(tampil_progress_html())
    with out_jadwal: clear_output(); display(buat_jadwal())

def tambah(btn):
    with out_status:
        clear_output()
        if not nama.value.strip():
            print("Nama harus diisi")
            return
        dl = deadline.value.isoformat() if deadline.value else None
        add_task(nama.value.strip(), waktu.value, dl, prioritas.value, kategori.value)
        print(f"✅ Tugas '{nama.value}' ditambahkan")
        nama.value = ""; waktu.value=30; prioritas.value='Medium'; deadline.value=None; kategori.value=""
        refresh()

def hapus_semua(btn):
    with out_status:
        clear_output()
        clear_all_tasks()
        print("🗑 Semua tugas dihapus")
        refresh()

btn_tambah.on_click(tambah)
btn_reset.on_click(hapus_semua)

# Tandai selesai
id_selesai = widgets.IntText(description="ID Tugas:", value=1)
btn_selesai = widgets.Button(description="✔️ Tandai Selesai", button_style='success')

def selesaikan(btn):
    with out_status:
        clear_output()
        tid = id_selesai.value
        success, msg = complete_task_by_id(tid)
        if success:
            print(f"✅ {msg} - Tugas ID {tid} selesai!")
            refresh()
        else:
            print(f"❌ {msg} - ID {tid} gagal diupdate. Pastikan ID benar dan status masih pending.")
        id_selesai.value = 1

btn_selesai.on_click(selesaikan)

# Layout
display(HTML("<h1 style='color:#ffd966'>✨ ChronoScore - Kelompok 6 ✨</h1>"))
display(HTML("<h3>✏️ Tambah Tugas Baru</h3>"))
display(widgets.VBox([nama, waktu, prioritas, deadline, kategori, widgets.HBox([btn_tambah, btn_reset])]))
display(HTML("<hr><h3>📋 Daftar Tugas (Lihat ID untuk menandai)</h3>"))
display(out_tabel)
display(HTML("<hr><h3>✅ Tandai Tugas Selesai</h3>"))
display(widgets.HBox([id_selesai, btn_selesai]))
display(HTML("<hr><h3>📊 Progress & Jadwal</h3>"))
display(out_progress)
display(out_jadwal)
display(out_status)

refresh()

Output()

Output()

Output()

Output()

In [3]:
import sqlite3
from datetime import datetime, timedelta
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd

# PALLETE WARNA
BG_MAIN = "#0a0a0a"          # background utama hitam
CARD_BG = "#F5F0E9"          # Swan Wing (krem lembut)
CARD_TEXT = "#1a1a1a"
ACCENT = "#3C507D"           # Sapphire (biru) tetap untuk border/aksen lain
ACCENT_LIGHT = "#112250"     # Royal Blue (biru) tetap untuk aksen lain
ACCENT_WARM = "#E0C58F"      # Quicksand
DANGER = "#09CBC2"           # Shellstone
HEADER_BG = "#210100"        # Bistre
HEADER_TEXT = "#FECE79"      # Butter

# DATABASE
DB_NAME = "tasks.db"

def init_db():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute('''CREATE TABLE IF NOT EXISTS tasks (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        estimated_minutes INTEGER,
        deadline TEXT,
        priority TEXT,
        category TEXT,
        status TEXT DEFAULT 'pending',
        created_at TEXT DEFAULT CURRENT_TIMESTAMP,
        completed_at TEXT
    )''')
    conn.commit()
    conn.close()

init_db()

# FUNGSI DASAR
def add_task(name, minutes, deadline, priority, category):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute('''INSERT INTO tasks (name, estimated_minutes, deadline, priority, category, status)
                 VALUES (?,?,?,?,?, 'pending')''', (name, minutes, deadline, priority, category))
    conn.commit()
    conn.close()

def get_all_tasks_df():
    conn = sqlite3.connect(DB_NAME)
    df = pd.read_sql_query("SELECT id, name, estimated_minutes, deadline, priority, category, status FROM tasks", conn)
    conn.close()
    return df

def clear_all_tasks():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("DELETE FROM tasks")
    conn.commit()
    conn.close()

def complete_task_by_id(task_id):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    try:
        c.execute("SELECT status FROM tasks WHERE id=?", (task_id,))
        row = c.fetchone()
        if row is None:
            return False, "ID tidak ditemukan"
        if row[0] != 'pending':
            return False, "Tugas sudah selesai"
        now = datetime.now().isoformat()
        c.execute("UPDATE tasks SET status='completed', completed_at=? WHERE id=? AND status='pending'",
                  (now, task_id))
        conn.commit()
        return c.rowcount > 0, "Berhasil"
    except Exception as e:
        return False, str(e)
    finally:
        conn.close()

# DATA CONTOH
def insert_sample_tasks():
    if get_all_tasks_df().empty:
        sample = [
            ("Belajar UAP", 100, "2026-06-12", "High", "Akademik"),
            ("Menulis Ilmiah", 60, "2026-06-10", "High", "Akademik"),
            ("Jogging", 35, "2026-06-11", "Medium", "Pribadi"),
            ("Video Resume FPO", 90, "2026-06-13", "Medium", "Non akademik"),
            ("Notulensi", 30, "2026-06-09", "Medium", "Organisasi")
        ]
        for nama, menit, deadline, prioritas, kategori in sample:
            add_task(nama, menit, deadline, prioritas, kategori)
        print("✅ Data contoh ditambahkan.\n")

insert_sample_tasks()

# WARNA BAIGROUND
def get_css():
    return f"""
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@400;700&family=Poppins:wght@300;400;600&display=swap');
        .jawa-container {{
            font-family: 'Poppins', sans-serif;
            background: #0a0a0a;
            color: #e6e6e6;
            padding: 20px;
            border-radius: 20px;
            box-shadow: 0 0 20px rgba(0,0,0,0.5);
            border: 1px solid {ACCENT};
        }}
        .jawa-title {{
            font-family: 'Playfair Display', serif;
            font-size: 5em;
            font-weight: 800;
            text-align: center;
            color: white;
            text-shadow: 0 0 10px #3C507D, 0 0 20px #112250, 0 0 30px rgba(60,80,125,0.5);
            letter-spacing: 3px;
            margin-bottom: 10px;
            background: none;
            -webkit-background-clip: unset;
            background-clip: unset;
        }}
        .jawa-sub {{
            text-align: center;
            font-style: italic;
            color: {CARD_BG};
            margin-bottom: 20px;
            border-top: 1px solid {ACCENT};
            border-bottom: 1px solid {ACCENT};
            display: inline-block;
            padding: 5px 20px;
        }}
        .jawa-card {{
            background: {CARD_BG};
            border-radius: 15px;
            padding: 15px;
            margin: 15px 0;
            border-left: 4px solid {ACCENT};
            box-shadow: 0 4px 8px rgba(0,0,0,0.5);
            color: {CARD_TEXT};
        }}
        .jawa-card h3 {{
            color: {HEADER_BG};
            font-weight: bold;
            margin-top: 0;
        }}
        .jawa-button {{
            background: linear-gradient(90deg, {HEADER_BG}, #1a1a1a);
            color: {CARD_BG};
            border: 1px solid {ACCENT};
            border-radius: 30px;
            padding: 8px 20px;
            font-weight: bold;
            transition: 0.3s;
        }}
        .jawa-button:hover {{
            background: linear-gradient(90deg, {ACCENT}, {ACCENT_LIGHT});
            color: #0a0a0a;
            border-color: {CARD_BG};
            transform: scale(1.02);
        }}
        .jawa-table {{
            width: 100%;
            border-collapse: collapse;
            font-size: 14px;
            background: #1e1e1e;
            color: #e6e6e6;
        }}
        .jawa-table th {{
            background: {HEADER_BG};
            color: {HEADER_TEXT};
            padding: 12px;
            font-weight: 600;
            border-bottom: 2px solid {ACCENT};
        }}
        .jawa-table td {{
            padding: 10px;
            border-bottom: 1px solid #444;
        }}
         .jawa-progress {{
            border: 1px solid {ACCENT};
            border-radius: 30px;
            background: #1a1a1a;
            overflow: hidden;
        }}
        .jawa-progress-bar {{
            background: linear-gradient(90deg, #E6A341, #FECE79) !important;
            height: 30px;
            border-radius: 30px;
            text-align: center;
            line-height: 30px;
            color: black;
            font-weight: bold;
            transition: width 0.5s;
        }}
        .jawa-badge-medium {{
            background: #FECE79 !important;
            color: #1a1a1a !important;
            padding: 2px 8px;
            border-radius: 20px;
        }}
        .jawa-badge-high {{
            background: #8C0902;
            color: {CARD_BG};
            padding: 2px 8px;
            border-radius: 20px;
        }}
        .jawa-badge-low {{
            background: {HEADER_BG};
            color: {CARD_BG};
            padding: 2px 8px;
            border-radius: 20px;
        }}
        .widget-slider .slider .thumb {{
            background: #E6A341 !important;
        }}
        .widget-slider .slider .track-highlight {{
            background: #FECE79 !important;
        }}
        @keyframes blinkGold {{
            0% {{ color: {CARD_BG}; text-shadow: 0 0 0px; }}
            100% {{ color: {ACCENT_LIGHT}; text-shadow: 0 0 10px {ACCENT_LIGHT}; }}
        }}
        .blink-text {{
            animation: blinkGold 0.8s infinite alternate;
            font-weight: bold;
        }}
        @keyframes bounce {{
            from {{ transform: translateY(0px); }}
            to {{ transform: translateY(-15px); }}
        }}
        .jawa-card label, .jawa-card .widget-label {{
            color: {CARD_TEXT} !important;
        }}
        .jawa-card .widget-text input, .jawa-card .widget-dropdown select, .jawa-card .widget-inttext input {{
            background: #fff7e8;
            color: #2c2c2c;
        }}
    </style>
    """

# DATA DALAM TABEL
def tampilkan_tabel():
    df = get_all_tasks_df()
    if df.empty:
        return HTML("<p class='blink-text'>📭 Belum ada tugas. Silakan tambah!</p>")
    df_display = df.copy()
    df_display['status'] = df_display['status'].apply(lambda x: '⏳ Pending' if x=='pending' else '✅ Selesai')
    df_display.columns = ['ID', 'Nama Tugas', 'Menit', 'Deadline', 'Prioritas', 'Kategori', 'Status']

    html_table = "<table class='jawa-table'>"
    html_table += "<tr>" + "".join([f"<th>{col}</th>" for col in df_display.columns]) + "</tr>"
    for _, row in df_display.iterrows():
        if row['Prioritas'] == 'High':
            prio_badge = "<span class='jawa-badge-high'>High</span>"
            bg_row = "style='background-color: #2c1a1a'"
        elif row['Prioritas'] == 'Medium':
            prio_badge = "<span class='jawa-badge-medium'>Medium</span>"
            bg_row = "style='background-color: #2a2418'"
        else:
            prio_badge = "<span class='jawa-badge-low'>Low</span>"
            bg_row = "style='background-color: #1a2a1a'"
        html_table += f"<tr {bg_row}>"
        html_table += f"<td>{row['ID']}</td>"
        html_table += f"<td>{row['Nama Tugas']}</td>"
        html_table += f"<td>{row['Menit']}</td>"
        html_table += f"<td>{row['Deadline'] or '-'}</td>"
        html_table += f"<td>{prio_badge}</td>"
        html_table += f"<td>{row['Kategori']}</td>"
        html_table += f"<td>{row['Status']}</td>"
        html_table += "</tr>"
    html_table += "</table>"
    return HTML(get_css() + html_table)

# PROGRESS PENGERJAAN
def tampil_progress_html():
    df = get_all_tasks_df()
    if df.empty:
        return HTML("<p>Belum ada tugas.</p>")
    total = len(df)
    selesai = len(df[df['status']=='completed'])
    pending = total - selesai
    percent = (selesai / total) * 100 if total > 0 else 0

    html = f"""
    <div class='jawa-progress'>
        <div class='jawa-progress-bar' style='width:{percent}%;'>
            {percent:.1f}%
        </div>
    </div>
    <p style='margin-top:10px'>✨ Selesai: {selesai} | Pending: {pending} | Total: {total}</p>
    """
    if percent == 100:
        html += """
        <div style="text-align:center; font-size:40px; animation: bounce 0.5s infinite alternate;">
            😊🌾🎉✨🥳
        </div>
        <div class='blink-text' style='text-align:center; font-size:24px;'>
            🌟 SELAMAT! SEMUA TUGAS ANDA SELESAI! 🌟
        </div>
        <script>
        (function() {
            if (typeof canvasConfetti === 'undefined') {
                var script = document.createElement('script');
                script.src = 'https://cdn.jsdelivr.net/npm/canvas-confetti@1';
                script.onload = function() {
                    canvasConfetti({particleCount: 300, spread: 100, origin: {y: 0.6}, colors: ['#E6A341', '#FECE79']});
                };
                document.head.appendChild(script);
            } else {
                canvasConfetti({particleCount: 300, spread: 100, origin: {y: 0.6}, colors: ['#E6A341', '#FECE79']});
            }
        })();
        </script>
        """
    return HTML(get_css() + html)

# DATA JADWAL
def buat_jadwal():
    df = get_all_tasks_df()
    pending = df[df['status']=='pending']
    if pending.empty:
        return HTML("<p>🎉 Semua tugas selesai. Tidak ada jadwal hari ini.</p>")
    total_menit = 8*60
    used = 0
    mulai = datetime.strptime("09:00", "%H:%M")
    html = f"<h3 style='color:{HEADER_BG}'>📅 Jadwal Hari Ini (8 jam, mulai jam 09:00)</h3><ul style='list-style:none; padding-left:0'>"
    for _, row in pending.iterrows():
        dur = row['estimated_minutes']
        if used + dur <= total_menit:
            selesai = mulai + timedelta(minutes=dur)
            warna = "#8C0902" if row['priority']=='High' else "#E6A341" if row['priority']=='Medium' else "#210100"
            html += f"<li style='background:{warna}; color:white; margin:8px; padding:10px; border-radius:15px; border-left:5px solid {CARD_BG}'>"
            html += f"<b>{mulai.strftime('%H:%M')} - {selesai.strftime('%H:%M')}</b> | {row['name']} ({dur} menit) | Prioritas {row['priority']}</li>"
            mulai = selesai
            used += dur
        else:
            html += f"<li style='background:#333; color:#ccc; margin:8px; padding:10px; border-radius:15px'>⚠️ {row['name']} tidak muat → besok</li>"
    html += f"</ul><p>⏱ Total terpakai: {used/60:.1f} jam</p>"
    return HTML(get_css() + html)

# INPUT DATA
nama_input = widgets.Text(placeholder="Nama tugas", description="📝 Nama:", layout=widgets.Layout(width='500px'))
waktu_input = widgets.IntSlider(min=5, max=500, step=5, value=30, description="⏱ Menit:", layout=widgets.Layout(width='500px'))
prioritas_input = widgets.Dropdown(options=['High','Medium','Low'], value='Medium', description="⚡ Prioritas:", layout=widgets.Layout(width='500px'))
deadline_input = widgets.DatePicker(description="📅 Deadline:", layout=widgets.Layout(width='500px'))
kategori_input = widgets.Text(placeholder="Kategori", description="📂 Kategori:", layout=widgets.Layout(width='500px'))

btn_tambah = widgets.Button(description="➕ TAMBAH TUGAS", layout=widgets.Layout(width='300px'))
btn_reset = widgets.Button(description="🗑 HAPUS SEMUA", layout=widgets.Layout(width='300px'))

btn_tambah.add_class('jawa-button')
btn_reset.add_class('jawa-button')

out_tabel = widgets.Output()
out_progress = widgets.Output()
out_jadwal = widgets.Output()
out_status = widgets.Output()

def refresh_all():
    with out_tabel:
        clear_output(wait=True)
        display(HTML(get_css()))
        display(tampilkan_tabel())
    with out_progress:
        clear_output(wait=True)
        display(HTML(get_css()))
        display(tampil_progress_html())
    with out_jadwal:
        clear_output(wait=True)
        display(HTML(get_css()))
        display(buat_jadwal())

def tambah(btn):
    with out_status:
        clear_output()
        nama = nama_input.value.strip()
        if not nama:
            print("⚠️ Nama tugas harus diisi!")
            return
        dl = deadline_input.value.isoformat() if deadline_input.value else None
        add_task(nama, waktu_input.value, dl, prioritas_input.value, kategori_input.value)
        print(f"✅ Tugas '{nama}' berhasil ditambahkan!")
        nama_input.value = ""
        waktu_input.value = 30
        prioritas_input.value = "Medium"
        deadline_input.value = None
        kategori_input.value = ""
        refresh_all()

def hapus_semua(btn):
    with out_status:
        clear_output()
        clear_all_tasks()
        print("🗑 Semua tugas telah dihapus.")
        refresh_all()
        insert_sample_tasks()
        refresh_all()

btn_tambah.on_click(tambah)
btn_reset.on_click(hapus_semua)

# TANDAI SELESAI
id_selesai = widgets.IntText(description="ID Tugas:", value=1, layout=widgets.Layout(width='150px'))
btn_selesai = widgets.Button(description="✔️ TANDAI SELESAI", layout=widgets.Layout(width='180px'))
btn_selesai.add_class('jawa-button')

def selesaikan(btn):
    with out_status:
        clear_output()
        tid = id_selesai.value
        sukses, pesan = complete_task_by_id(tid)
        if sukses:
            print(f"✅ Tugas ID {tid} telah diselesaikan!")
            refresh_all()
        else:
            print(f"❌ Gagal: {pesan}. Cek ID {tid} dan pastikan status masih Pending.")
        id_selesai.value = 1

btn_selesai.on_click(selesaikan)

# TAMPILAN UTAMA
display(HTML(get_css()))
display(HTML(f"""
<div class='jawa-container'>
    <div class='jawa-title'>ChronoScore</div>
    <div style='text-align:center;'><span class='jawa-sub'>TO DO LIST BY KELOMPOK 6</span></div>
"""))

# 1. Form tambah
display(HTML("<div class='jawa-card'><h3 style='text-align:center'>✍️ TAMBAH TUGAS BARU</h3><div style='display: flex; justify-content: center;'>"))
display(widgets.VBox([
    nama_input, waktu_input, prioritas_input, deadline_input, kategori_input,
    widgets.HBox([btn_tambah, btn_reset])
], layout=widgets.Layout(align_items='center')))
display(HTML("</div></div>"))


# 2. Tabel
display(HTML("<div class='jawa-card'><h3>📋 DAFTAR TUGAS</h3>"))
display(out_tabel)
display(HTML("</div>"))

# 3. Tandai selesai
display(HTML("<div class='jawa-card'><h3>✅ TANDAI TUGAS SELESAI</h3>"))
display(widgets.HBox([id_selesai, btn_selesai]))
display(HTML("</div>"))

# 4. Progress & Jadwal
display(HTML("<div class='jawa-card'><h3>📊 PROGRESS & JADWAL</h3>"))
display(out_progress)
display(out_jadwal)
display(HTML("</div>"))

display(HTML("</div>"))
display(out_status)

refresh_all()

Output()

Output()

Output()

Output()

In [4]:
import sqlite3
from datetime import datetime, timedelta, date
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd

# PALLETE WARNA
BG_MAIN = "#0a0a0a"
CARD_BG = "#F5F0E9"
CARD_TEXT = "#1a1a1a"
ACCENT = "#3C507D"
ACCENT_LIGHT = "#112250"
ACCENT_WARM = "#E0C58F"
DANGER = "#09CBC2"
HEADER_BG = "#210100"
HEADER_TEXT = "#FECE79"

# DATABASE
DB_NAME = "tasks.db"

def init_db():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute('''CREATE TABLE IF NOT EXISTS tasks (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        estimated_minutes INTEGER,
        deadline TEXT,
        priority TEXT,
        category TEXT,
        difficulty INTEGER DEFAULT 3,
        status TEXT DEFAULT 'pending',
        created_at TEXT DEFAULT CURRENT_TIMESTAMP,
        completed_at TEXT
    )''')
    # Tambah kolom difficulty jika tabel lama belum punya
    try:
        c.execute("ALTER TABLE tasks ADD COLUMN difficulty INTEGER DEFAULT 3")
    except:
        pass
    conn.commit()
    conn.close()

init_db()

# ─── RUMUS SKOR PRIORITAS ────────────────────────────────────────────────────
# Skor = (difficulty × 20) + deadline_urgency
# deadline_urgency: 100 jika sudah lewat, berkurang 10/hari, min 0
# Hasil: 0–200, lalu dibagi 2 → 0–100 skala
# Priority otomatis: High ≥ 65 | Medium 35–64 | Low < 35
def hitung_skor(difficulty, deadline_str):
    """
    difficulty : int 1–5
    deadline_str: 'YYYY-MM-DD' atau None
    returns: skor (int 0–100), priority_label (str)
    """
    difficulty_score = (difficulty / 5) * 50          # maks 50 poin

    if deadline_str:
        try:
            dl = datetime.strptime(deadline_str, "%Y-%m-%d").date()
            hari_tersisa = (dl - date.today()).days
            # Makin dekat/lewat deadline → skor makin tinggi
            if hari_tersisa <= 0:
                urgency_score = 50                     # sudah lewat/hari ini = maks
            elif hari_tersisa <= 3:
                urgency_score = 40
            elif hari_tersisa <= 7:
                urgency_score = 25
            elif hari_tersisa <= 14:
                urgency_score = 15
            else:
                urgency_score = 5
        except:
            urgency_score = 0
    else:
        urgency_score = 0

    skor = round(difficulty_score + urgency_score)

    if skor >= 65:
        priority = "High"
    elif skor >= 35:
        priority = "Medium"
    else:
        priority = "Low"

    return skor, priority

# FUNGSI DASAR
def add_task(name, minutes, deadline, category, difficulty):
    skor, priority = hitung_skor(difficulty, deadline)
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute('''INSERT INTO tasks (name, estimated_minutes, deadline, priority, category, difficulty, status)
                 VALUES (?,?,?,?,?,?,'pending')''',
              (name, minutes, deadline, priority, category, difficulty))
    conn.commit()
    conn.close()

def get_all_tasks_df():
    conn = sqlite3.connect(DB_NAME)
    df = pd.read_sql_query(
        "SELECT id, name, estimated_minutes, deadline, priority, category, difficulty, status FROM tasks", conn)
    conn.close()
    return df

def clear_all_tasks():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("DELETE FROM tasks")
    conn.commit()
    conn.close()

def complete_task_by_id(task_id):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    try:
        c.execute("SELECT status FROM tasks WHERE id=?", (task_id,))
        row = c.fetchone()
        if row is None:
            return False, "ID tidak ditemukan"
        if row[0] != 'pending':
            return False, "Tugas sudah selesai"
        now = datetime.now().isoformat()
        c.execute("UPDATE tasks SET status='completed', completed_at=? WHERE id=? AND status='pending'",
                  (now, task_id))
        conn.commit()
        return c.rowcount > 0, "Berhasil"
    except Exception as e:
        return False, str(e)
    finally:
        conn.close()

# Recalculate semua skor (misalnya kalau hari berganti)
def recalc_all_scores():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT id, difficulty, deadline FROM tasks WHERE status='pending'")
    rows = c.fetchall()
    for tid, diff, dl in rows:
        skor, prio = hitung_skor(diff or 3, dl)
        c.execute("UPDATE tasks SET priority=? WHERE id=?", (prio, tid))
    conn.commit()
    conn.close()

# DATA CONTOH
def insert_sample_tasks():
    if get_all_tasks_df().empty:
        sample = [
            ("Belajar UAP",      100, "2026-06-12", "Akademik",     4),
            ("Menulis Ilmiah",    60, "2026-06-10", "Akademik",     3),
            ("Jogging",           35, "2026-06-11", "Pribadi",      1),
            ("Video Resume FPO",  90, "2026-06-13", "Non akademik", 3),
            ("Notulensi",         30, "2026-06-09", "Organisasi",   2),
        ]
        for nama, menit, deadline, kategori, difficulty in sample:
            add_task(nama, menit, deadline, kategori, difficulty)
        print("✅ Data contoh ditambahkan.\n")

insert_sample_tasks()
recalc_all_scores()

# ─── CSS ─────────────────────────────────────────────────────────────────────
def get_css():
    return f"""
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@400;700&family=Poppins:wght@300;400;600&display=swap');
        .jawa-container {{
            font-family: 'Poppins', sans-serif;
            background: #0a0a0a;
            color: #e6e6e6;
            padding: 20px;
            border-radius: 20px;
            box-shadow: 0 0 20px rgba(0,0,0,0.5);
            border: 1px solid {ACCENT};
        }}
        .jawa-title {{
            font-family: 'Playfair Display', serif;
            font-size: 5em;
            font-weight: 800;
            text-align: center;
            color: white;
            text-shadow: 0 0 10px #3C507D, 0 0 20px #112250, 0 0 30px rgba(60,80,125,0.5);
            letter-spacing: 3px;
            margin-bottom: 10px;
        }}
        .jawa-sub {{
            text-align: center;
            font-style: italic;
            color: {CARD_BG};
            margin-bottom: 20px;
            border-top: 1px solid {ACCENT};
            border-bottom: 1px solid {ACCENT};
            display: inline-block;
            padding: 5px 20px;
        }}
        .jawa-card {{
            background: {CARD_BG};
            border-radius: 15px;
            padding: 15px;
            margin: 15px 0;
            border-left: 4px solid {ACCENT};
            box-shadow: 0 4px 8px rgba(0,0,0,0.5);
            color: {CARD_TEXT};
        }}
        .jawa-card h3 {{
            color: {HEADER_BG};
            font-weight: bold;
            margin-top: 0;
        }}
        .jawa-button {{
            background: linear-gradient(90deg, {HEADER_BG}, #1a1a1a);
            color: {CARD_BG};
            border: 1px solid {ACCENT};
            border-radius: 30px;
            padding: 8px 20px;
            font-weight: bold;
            transition: 0.3s;
        }}
        .jawa-button:hover {{
            background: linear-gradient(90deg, {ACCENT}, {ACCENT_LIGHT});
            color: #0a0a0a;
            border-color: {CARD_BG};
            transform: scale(1.02);
        }}
        .jawa-table {{
            width: 100%;
            border-collapse: collapse;
            font-size: 13px;
            background: #1e1e1e;
            color: #e6e6e6;
        }}
        .jawa-table th {{
            background: {HEADER_BG};
            color: {HEADER_TEXT};
            padding: 12px;
            font-weight: 600;
            border-bottom: 2px solid {ACCENT};
            white-space: nowrap;
        }}
        .jawa-table td {{
            padding: 10px;
            border-bottom: 1px solid #444;
        }}
        .jawa-progress {{
            border: 1px solid {ACCENT};
            border-radius: 30px;
            background: #1a1a1a;
            overflow: hidden;
        }}
        .jawa-progress-bar {{
            background: linear-gradient(90deg, #E6A341, #FECE79) !important;
            height: 30px;
            border-radius: 30px;
            text-align: center;
            line-height: 30px;
            color: black;
            font-weight: bold;
            transition: width 0.5s;
        }}
        .jawa-badge-medium {{
            background: #FECE79 !important;
            color: #1a1a1a !important;
            padding: 2px 8px;
            border-radius: 20px;
            font-size: 12px;
        }}
        .jawa-badge-high {{
            background: #8C0902;
            color: {CARD_BG};
            padding: 2px 8px;
            border-radius: 20px;
            font-size: 12px;
        }}
        .jawa-badge-low {{
            background: {HEADER_BG};
            color: {CARD_BG};
            padding: 2px 8px;
            border-radius: 20px;
            font-size: 12px;
        }}
        /* Skor bar */
        .score-bar-wrap {{
            background: #333;
            border-radius: 10px;
            width: 80px;
            height: 14px;
            display: inline-block;
            vertical-align: middle;
            margin-left: 5px;
        }}
        .score-bar-fill {{
            height: 14px;
            border-radius: 10px;
        }}
        .rank-badge {{
            display: inline-block;
            width: 26px;
            height: 26px;
            border-radius: 50%;
            text-align: center;
            line-height: 26px;
            font-weight: bold;
            font-size: 13px;
        }}
        .rank-1 {{ background: #FFD700; color: #1a1a1a; }}
        .rank-2 {{ background: #C0C0C0; color: #1a1a1a; }}
        .rank-3 {{ background: #CD7F32; color: #fff; }}
        .rank-other {{ background: #3C507D; color: #fff; }}
        .blink-text {{
            animation: blinkGold 0.8s infinite alternate;
            font-weight: bold;
        }}
        @keyframes blinkGold {{
            0% {{ color: {CARD_BG}; }}
            100% {{ color: {ACCENT_LIGHT}; text-shadow: 0 0 10px {ACCENT_LIGHT}; }}
        }}
        @keyframes bounce {{
            from {{ transform: translateY(0px); }}
            to {{ transform: translateY(-15px); }}
        }}
        .jawa-card label, .jawa-card .widget-label {{ color: {CARD_TEXT} !important; }}
        .jawa-card .widget-text input,
        .jawa-card .widget-dropdown select,
        .jawa-card .widget-inttext input {{
            background: #fff7e8;
            color: #2c2c2c;
        }}
        .formula-box {{
            background: #1a1a1a;
            border: 1px solid {ACCENT};
            border-radius: 12px;
            padding: 14px 18px;
            font-family: monospace;
            font-size: 13px;
            color: {HEADER_TEXT};
            margin: 10px 0;
            line-height: 1.8;
        }}
        .star-display {{
            color: #FFD700;
            font-size: 16px;
            letter-spacing: 2px;
        }}
        .diff-badge {{
            padding: 2px 8px;
            border-radius: 20px;
            font-size: 12px;
            font-weight: bold;
        }}
    </style>
    """

# ─── HELPER VISUAL ────────────────────────────────────────────────────────────
def bintang(n):
    return "★" * n + "☆" * (5 - n)

def score_color(skor):
    if skor >= 65: return "#c0392b"
    if skor >= 35: return "#e6a341"
    return "#27ae60"

def diff_color(d):
    colors = {1: "#27ae60", 2: "#2ecc71", 3: "#e6a341", 4: "#e67e22", 5: "#c0392b"}
    return colors.get(d, "#888")

# ─── TABEL ────────────────────────────────────────────────────────────────────
def tampilkan_tabel():
    df = get_all_tasks_df()
    if df.empty:
        return HTML("<p class='blink-text'>📭 Belum ada tugas. Silakan tambah!</p>")

    # Hitung skor tiap baris
    df['skor'] = df.apply(
        lambda r: hitung_skor(r['difficulty'] if r['difficulty'] else 3, r['deadline'])[0], axis=1)
    df['priority'] = df.apply(
        lambda r: hitung_skor(r['difficulty'] if r['difficulty'] else 3, r['deadline'])[1], axis=1)

    # Rank hanya untuk pending, diurutkan dari skor tertinggi
    pending_df = df[df['status'] == 'pending'].copy()
    pending_df = pending_df.sort_values('skor', ascending=False).reset_index(drop=True)
    pending_df['rank'] = range(1, len(pending_df) + 1)

    completed_df = df[df['status'] == 'completed'].copy()
    completed_df['rank'] = None

    df_sorted = pd.concat([pending_df, completed_df])

    html = "<table class='jawa-table'>"
    html += "<tr>"
    for col in ['Rank', 'ID', 'Nama Tugas', 'Menit', 'Deadline', 'Kesulitan', 'Skor', 'Prioritas', 'Kategori', 'Status']:
        html += f"<th>{col}</th>"
    html += "</tr>"

    for _, row in df_sorted.iterrows():
        is_done = row['status'] == 'completed'
        row_style = "style='opacity:0.5'" if is_done else ""
        skor = row['skor']
        diff = row['difficulty'] if row['difficulty'] else 3
        prio = row['priority']

        # Rank badge
        if pd.isna(row['rank']) or is_done:
            rank_html = "<td style='text-align:center'>—</td>"
        else:
            r = int(row['rank'])
            cls = {1: 'rank-1', 2: 'rank-2', 3: 'rank-3'}.get(r, 'rank-other')
            rank_html = f"<td style='text-align:center'><span class='rank-badge {cls}'>{r}</span></td>"

        # Priority badge
        if prio == 'High':
            prio_html = "<span class='jawa-badge-high'>🔴 High</span>"
            bg = "background-color:#2c1a1a"
        elif prio == 'Medium':
            prio_html = "<span class='jawa-badge-medium'>🟡 Medium</span>"
            bg = "background-color:#2a2418"
        else:
            prio_html = "<span class='jawa-badge-low'>🟢 Low</span>"
            bg = "background-color:#1a2a1a"

        if is_done:
            bg = "background-color:#1a1a1a"

        # Skor bar
        bar_color = score_color(skor)
        skor_html = f"""
            <span style='font-weight:bold;color:{bar_color}'>{skor}</span>
            <span class='score-bar-wrap'>
                <span class='score-bar-fill' style='width:{skor}%;background:{bar_color}'></span>
            </span>"""

        # Difficulty stars
        diff_html = f"<span class='star-display' style='color:{diff_color(diff)}'>{bintang(diff)}</span>"

        status_html = '✅ Selesai' if is_done else '⏳ Pending'

        html += f"<tr style='{bg}' {row_style}>"
        html += rank_html
        html += f"<td>{int(row['id'])}</td>"
        html += f"<td><b>{row['name']}</b></td>"
        html += f"<td>{row['estimated_minutes']}</td>"
        html += f"<td>{row['deadline'] or '—'}</td>"
        html += f"<td>{diff_html}</td>"
        html += f"<td>{skor_html}</td>"
        html += f"<td>{prio_html}</td>"
        html += f"<td>{row['category']}</td>"
        html += f"<td>{status_html}</td>"
        html += "</tr>"

    html += "</table>"
    return HTML(get_css() + html)

# ─── FORMULA INFO ─────────────────────────────────────────────────────────────
def tampil_formula():
    html = f"""
    <div class='formula-box'>
        <b style='color:{HEADER_TEXT};font-size:15px'>⚙️ Rumus Penghitungan Skor Prioritas</b><br><br>
        <span style='color:#aaa'>Skor (0–100) = Skor Kesulitan + Skor Urgensi Deadline</span><br><br>
        <b>Skor Kesulitan</b> = (tingkat_kesulitan / 5) × 50 &nbsp;→ maks 50 poin<br>
        &nbsp;&nbsp;&nbsp;⭐ Kesulitan 1 → +10 | ⭐⭐ 2 → +20 | ⭐⭐⭐ 3 → +30 | ⭐⭐⭐⭐ 4 → +40 | ⭐⭐⭐⭐⭐ 5 → +50<br><br>
        <b>Skor Urgensi Deadline</b>:<br>
        &nbsp;&nbsp;&nbsp;📛 Sudah lewat / hari ini → +50<br>
        &nbsp;&nbsp;&nbsp;🔴 1–3 hari lagi → +40<br>
        &nbsp;&nbsp;&nbsp;🟠 4–7 hari lagi → +25<br>
        &nbsp;&nbsp;&nbsp;🟡 8–14 hari lagi → +15<br>
        &nbsp;&nbsp;&nbsp;🟢 &gt; 14 hari lagi → +5<br><br>
        <b>Klasifikasi Prioritas Otomatis:</b><br>
        &nbsp;&nbsp;&nbsp;<span style='color:#c0392b'>🔴 High</span> = Skor ≥ 65 &nbsp;|&nbsp;
        <span style='color:#e6a341'>🟡 Medium</span> = Skor 35–64 &nbsp;|&nbsp;
        <span style='color:#27ae60'>🟢 Low</span> = Skor &lt; 35
    </div>
    """
    return HTML(get_css() + html)

# ─── PROGRESS ─────────────────────────────────────────────────────────────────
def tampil_progress_html():
    df = get_all_tasks_df()
    if df.empty:
        return HTML("<p>Belum ada tugas.</p>")
    total = len(df)
    selesai = len(df[df['status'] == 'completed'])
    pending = total - selesai
    percent = (selesai / total) * 100 if total > 0 else 0

    html = f"""
    <div class='jawa-progress'>
        <div class='jawa-progress-bar' style='width:{percent}%;'>
            {percent:.1f}%
        </div>
    </div>
    <p style='margin-top:10px'>✨ Selesai: {selesai} | Pending: {pending} | Total: {total}</p>
    """
    if percent == 100:
        html += """
        <div style="text-align:center; font-size:40px; animation: bounce 0.5s infinite alternate;">
            😊🌾🎉✨🥳
        </div>
        <div class='blink-text' style='text-align:center; font-size:24px;'>
            🌟 SELAMAT! SEMUA TUGAS ANDA SELESAI! 🌟
        </div>
        <script>
        (function() {
            if (typeof canvasConfetti === 'undefined') {
                var script = document.createElement('script');
                script.src = 'https://cdn.jsdelivr.net/npm/canvas-confetti@1';
                script.onload = function() {
                    canvasConfetti({particleCount: 300, spread: 100, origin: {y: 0.6}, colors: ['#E6A341', '#FECE79']});
                };
                document.head.appendChild(script);
            } else {
                canvasConfetti({particleCount: 300, spread: 100, origin: {y: 0.6}, colors: ['#E6A341', '#FECE79']});
            }
        })();
        </script>
        """
    return HTML(get_css() + html)

# ─── JADWAL ───────────────────────────────────────────────────────────────────
def buat_jadwal():
    df = get_all_tasks_df()
    pending = df[df['status'] == 'pending'].copy()
    if pending.empty:
        return HTML("<p>🎉 Semua tugas selesai. Tidak ada jadwal hari ini.</p>")

    # Urutkan berdasarkan skor prioritas (tertinggi duluan)
    pending['skor'] = pending.apply(
        lambda r: hitung_skor(r['difficulty'] if r['difficulty'] else 3, r['deadline'])[0], axis=1)
    pending = pending.sort_values('skor', ascending=False)

    total_menit = 8 * 60
    used = 0
    mulai = datetime.strptime("09:00", "%H:%M")
    html = f"<h3 style='color:{HEADER_BG}'>📅 Jadwal Hari Ini (8 jam, mulai 09:00) — diurutkan by Skor</h3>"
    html += "<ul style='list-style:none; padding-left:0'>"
    for _, row in pending.iterrows():
        dur = row['estimated_minutes']
        skor = row['skor']
        warna = "#8C0902" if skor >= 65 else "#E6A341" if skor >= 35 else "#210100"
        if used + dur <= total_menit:
            selesai = mulai + timedelta(minutes=dur)
            html += f"""<li style='background:{warna}; color:white; margin:8px; padding:10px;
                         border-radius:15px; border-left:5px solid {CARD_BG}'>
                         <b>{mulai.strftime('%H:%M')} – {selesai.strftime('%H:%M')}</b>
                         &nbsp;|&nbsp; {row['name']} ({dur} mnt)
                         &nbsp;|&nbsp; Skor: <b>{skor}</b>
                         &nbsp;|&nbsp; {bintang(row['difficulty'] if row['difficulty'] else 3)}
                         </li>"""
            mulai = selesai
            used += dur
        else:
            html += f"<li style='background:#333; color:#ccc; margin:8px; padding:10px; border-radius:15px'>⚠️ {row['name']} tidak muat hari ini → jadwalkan besok</li>"
    html += f"</ul><p>⏱ Total terpakai: {used/60:.1f} jam dari 8 jam</p>"
    return HTML(get_css() + html)

# ─── WIDGETS ──────────────────────────────────────────────────────────────────
nama_input      = widgets.Text(placeholder="Nama tugas", description="📝 Nama:",
                               layout=widgets.Layout(width='500px'))
waktu_input     = widgets.IntSlider(min=5, max=500, step=5, value=30, description="⏱ Menit:",
                                    layout=widgets.Layout(width='500px'))
deadline_input  = widgets.DatePicker(description="📅 Deadline:",
                                     layout=widgets.Layout(width='500px'))
kategori_input  = widgets.Text(placeholder="Kategori", description="📂 Kategori:",
                                layout=widgets.Layout(width='500px'))

# ★ Difficulty rating baru (1–5)
difficulty_input = widgets.RadioButtons(
    options=[(f'{bintang(i)}  ({i})', i) for i in range(1, 6)],
    value=3,
    description="🎯 Kesulitan:",
    layout=widgets.Layout(width='500px')
)

# Preview skor live
out_score_preview = widgets.Output()

def update_score_preview(*args):
    dl = deadline_input.value.isoformat() if deadline_input.value else None
    skor, prio = hitung_skor(difficulty_input.value, dl)
    bar_color = score_color(skor)
    prio_color = {"High": "#c0392b", "Medium": "#e6a341", "Low": "#27ae60"}[prio]
    with out_score_preview:
        clear_output(wait=True)
        display(HTML(get_css() + f"""
        <div style='background:#1a1a1a; border:1px solid {ACCENT}; border-radius:12px;
                    padding:12px 18px; display:inline-block; margin-top:5px;'>
            <span style='color:#aaa; font-size:13px'>Estimasi Skor: </span>
            <span style='font-size:22px; font-weight:bold; color:{bar_color}'>{skor}</span>
            <span class='score-bar-wrap' style='width:120px; vertical-align:middle; margin:0 10px'>
                <span class='score-bar-fill' style='width:{skor}%; background:{bar_color}'></span>
            </span>
            <span style='color:{prio_color}; font-weight:bold'>→ {prio}</span>
        </div>
        """))

difficulty_input.observe(update_score_preview, names='value')
deadline_input.observe(update_score_preview, names='value')

btn_tambah = widgets.Button(description="➕ TAMBAH TUGAS", layout=widgets.Layout(width='300px'))
btn_reset  = widgets.Button(description="🗑 HAPUS SEMUA",  layout=widgets.Layout(width='300px'))
btn_tambah.add_class('jawa-button')
btn_reset.add_class('jawa-button')

out_tabel   = widgets.Output()
out_progress = widgets.Output()
out_jadwal  = widgets.Output()
out_status  = widgets.Output()
out_formula = widgets.Output()

def refresh_all():
    recalc_all_scores()
    with out_tabel:
        clear_output(wait=True)
        display(HTML(get_css()))
        display(tampilkan_tabel())
    with out_progress:
        clear_output(wait=True)
        display(HTML(get_css()))
        display(tampil_progress_html())
    with out_jadwal:
        clear_output(wait=True)
        display(HTML(get_css()))
        display(buat_jadwal())
    with out_formula:
        clear_output(wait=True)
        display(tampil_formula())

def tambah(btn):
    with out_status:
        clear_output()
        nama = nama_input.value.strip()
        if not nama:
            print("⚠️ Nama tugas harus diisi!")
            return
        dl = deadline_input.value.isoformat() if deadline_input.value else None
        add_task(nama, waktu_input.value, dl, kategori_input.value, difficulty_input.value)
        skor, prio = hitung_skor(difficulty_input.value, dl)
        print(f"✅ '{nama}' ditambahkan! Skor: {skor} → Prioritas: {prio}")
        nama_input.value = ""
        waktu_input.value = 30
        deadline_input.value = None
        kategori_input.value = ""
        difficulty_input.value = 3
        refresh_all()
        update_score_preview()

def hapus_semua(btn):
    with out_status:
        clear_output()
        clear_all_tasks()
        print("🗑 Semua tugas telah dihapus.")
        refresh_all()
        insert_sample_tasks()
        refresh_all()

btn_tambah.on_click(tambah)
btn_reset.on_click(hapus_semua)

# Tandai selesai
id_selesai  = widgets.IntText(description="ID Tugas:", value=1, layout=widgets.Layout(width='150px'))
btn_selesai = widgets.Button(description="✔️ TANDAI SELESAI", layout=widgets.Layout(width='180px'))
btn_selesai.add_class('jawa-button')

def selesaikan(btn):
    with out_status:
        clear_output()
        tid = id_selesai.value
        sukses, pesan = complete_task_by_id(tid)
        if sukses:
            print(f"✅ Tugas ID {tid} telah diselesaikan!")
            refresh_all()
        else:
            print(f"❌ Gagal: {pesan}. Cek ID {tid} dan pastikan status masih Pending.")
        id_selesai.value = 1

btn_selesai.on_click(selesaikan)

# ─── TAMPILAN UTAMA ───────────────────────────────────────────────────────────
display(HTML(get_css()))
display(HTML(f"""
<div class='jawa-container'>
    <div class='jawa-title'>ChronoScore</div>
    <div style='text-align:center;'><span class='jawa-sub'>TO DO LIST BY KELOMPOK 6</span></div>
"""))

# 1. Rumus
display(HTML("<div class='jawa-card'><h3>⚙️ SISTEM SCORING PRIORITAS</h3>"))
display(out_formula)
display(HTML("</div>"))

# 2. Form tambah
display(HTML("<div class='jawa-card'><h3 style='text-align:center'>✍️ TAMBAH TUGAS BARU</h3>"
             "<div style='display:flex; justify-content:center;'>"))
display(widgets.VBox([
    nama_input, waktu_input, deadline_input, kategori_input,
    difficulty_input,
    out_score_preview,
    widgets.HBox([btn_tambah, btn_reset])
], layout=widgets.Layout(align_items='center')))
display(HTML("</div></div>"))

# 3. Tabel
display(HTML("<div class='jawa-card'><h3>📋 DAFTAR TUGAS (diurutkan by Skor)</h3>"))
display(out_tabel)
display(HTML("</div>"))

# 4. Tandai selesai
display(HTML("<div class='jawa-card'><h3>✅ TANDAI TUGAS SELESAI</h3>"))
display(widgets.HBox([id_selesai, btn_selesai]))
display(HTML("</div>"))

# 5. Progress & Jadwal
display(HTML("<div class='jawa-card'><h3>📊 PROGRESS & JADWAL</h3>"))
display(out_progress)
display(out_jadwal)
display(HTML("</div>"))

display(HTML("</div>"))
display(out_status)

refresh_all()
update_score_preview()

Output()

Output()

Output()

Output()

Output()

In [5]:
import sqlite3
from datetime import datetime, timedelta, date
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import io, base64

# Daftar kategori tetap
KATEGORI_LIST = ["Akademik", "Non Akademik", "Pribadi", "Organisasi"]
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import io, base64

# Daftar kategori tetap
KATEGORI_LIST = ["Akademik", "Non Akademik", "Pribadi", "Organisasi"]

# PALLETE WARNA
BG_MAIN = "#0a0a0a"
CARD_BG = "#F5F0E9"
CARD_TEXT = "#1a1a1a"
ACCENT = "#3C507D"
ACCENT_LIGHT = "#112250"
ACCENT_WARM = "#E0C58F"
DANGER = "#09CBC2"
HEADER_BG = "#210100"
HEADER_TEXT = "#FECE79"

# DATABASE
DB_NAME = "tasks.db"

def init_db():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute('''CREATE TABLE IF NOT EXISTS tasks (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        estimated_minutes INTEGER,
        deadline TEXT,
        priority TEXT,
        category TEXT,
        difficulty INTEGER DEFAULT 3,
        status TEXT DEFAULT 'pending',
        created_at TEXT DEFAULT CURRENT_TIMESTAMP,
        completed_at TEXT
    )''')
    # Tambah kolom difficulty jika tabel lama belum punya
    try:
        c.execute("ALTER TABLE tasks ADD COLUMN difficulty INTEGER DEFAULT 3")
    except:
        pass
    conn.commit()
    conn.close()

init_db()

# ─── RUMUS SKOR PRIORITAS ────────────────────────────────────────────────────
# Skor = (difficulty × 20) + deadline_urgency
# deadline_urgency: 100 jika sudah lewat, berkurang 10/hari, min 0
# Hasil: 0–200, lalu dibagi 2 → 0–100 skala
# Priority otomatis: High ≥ 65 | Medium 35–64 | Low < 35
def hitung_skor(difficulty, deadline_str):
    """
    difficulty : int 1–5
    deadline_str: 'YYYY-MM-DD' atau None
    returns: skor (int 0–100), priority_label (str)
    """
    difficulty_score = (difficulty / 5) * 50          # maks 50 poin

    if deadline_str:
        try:
            dl = datetime.strptime(deadline_str, "%Y-%m-%d").date()
            hari_tersisa = (dl - date.today()).days
            # Makin dekat/lewat deadline → skor makin tinggi
            if hari_tersisa <= 0:
                urgency_score = 50                     # sudah lewat/hari ini = maks
            elif hari_tersisa <= 3:
                urgency_score = 40
            elif hari_tersisa <= 7:
                urgency_score = 25
            elif hari_tersisa <= 14:
                urgency_score = 15
            else:
                urgency_score = 5
        except:
            urgency_score = 0
    else:
        urgency_score = 0

    skor = round(difficulty_score + urgency_score)

    if skor >= 65:
        priority = "High"
    elif skor >= 35:
        priority = "Medium"
    else:
        priority = "Low"

    return skor, priority

# FUNGSI DASAR
def add_task(name, minutes, deadline, category, difficulty):
    skor, priority = hitung_skor(difficulty, deadline)
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute('''INSERT INTO tasks (name, estimated_minutes, deadline, priority, category, difficulty, status)
                 VALUES (?,?,?,?,?,?,'pending')''',
              (name, minutes, deadline, priority, category, difficulty))
    conn.commit()
    conn.close()

def get_all_tasks_df():
    conn = sqlite3.connect(DB_NAME)
    df = pd.read_sql_query(
        "SELECT id, name, estimated_minutes, deadline, priority, category, difficulty, status FROM tasks", conn)
    conn.close()
    return df

def clear_all_tasks():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("DELETE FROM tasks")
    conn.commit()
    conn.close()

def complete_task_by_id(task_id):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    try:
        c.execute("SELECT status FROM tasks WHERE id=?", (task_id,))
        row = c.fetchone()
        if row is None:
            return False, "ID tidak ditemukan"
        if row[0] != 'pending':
            return False, "Tugas sudah selesai"
        now = datetime.now().isoformat()
        c.execute("UPDATE tasks SET status='completed', completed_at=? WHERE id=? AND status='pending'",
                  (now, task_id))
        conn.commit()
        return c.rowcount > 0, "Berhasil"
    except Exception as e:
        return False, str(e)
    finally:
        conn.close()

# Recalculate semua skor (misalnya kalau hari berganti)
def recalc_all_scores():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT id, difficulty, deadline FROM tasks WHERE status='pending'")
    rows = c.fetchall()
    for tid, diff, dl in rows:
        skor, prio = hitung_skor(diff or 3, dl)
        c.execute("UPDATE tasks SET priority=? WHERE id=?", (prio, tid))
    conn.commit()
    conn.close()

# DATA CONTOH
def insert_sample_tasks():
    if get_all_tasks_df().empty:
        sample = [
            ("Belajar UAP",      100, "2026-06-12", "Akademik",     4),
            ("Menulis Ilmiah",    60, "2026-06-10", "Akademik",     3),
            ("Jogging",           35, "2026-06-11", "Pribadi",      1),
            ("Video Resume FPO",  90, "2026-06-13", "Non akademik", 3),
            ("Notulensi",         30, "2026-06-09", "Organisasi",   2),
        ]
        for nama, menit, deadline, kategori, difficulty in sample:
            add_task(nama, menit, deadline, kategori, difficulty)
        print("✅ Data contoh ditambahkan.\n")

insert_sample_tasks()
recalc_all_scores()

# ─── CSS ─────────────────────────────────────────────────────────────────────
def get_css():
    return f"""
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@400;700&family=Poppins:wght@300;400;600&display=swap');
        .jawa-container {{
            font-family: 'Poppins', sans-serif;
            background: #0a0a0a;
            color: #e6e6e6;
            padding: 20px;
            border-radius: 20px;
            box-shadow: 0 0 20px rgba(0,0,0,0.5);
            border: 1px solid {ACCENT};
        }}
        .jawa-title {{
            font-family: 'Playfair Display', serif;
            font-size: 5em;
            font-weight: 800;
            text-align: center;
            color: white;
            text-shadow: 0 0 10px #3C507D, 0 0 20px #112250, 0 0 30px rgba(60,80,125,0.5);
            letter-spacing: 3px;
            margin-bottom: 10px;
        }}
        .jawa-sub {{
            text-align: center;
            font-style: italic;
            color: {CARD_BG};
            margin-bottom: 20px;
            border-top: 1px solid {ACCENT};
            border-bottom: 1px solid {ACCENT};
            display: inline-block;
            padding: 5px 20px;
        }}
        .jawa-card {{
            background: {CARD_BG};
            border-radius: 15px;
            padding: 15px;
            margin: 15px 0;
            border-left: 4px solid {ACCENT};
            box-shadow: 0 4px 8px rgba(0,0,0,0.5);
            color: {CARD_TEXT};
        }}
        .jawa-card h3 {{
            color: {HEADER_BG};
            font-weight: bold;
            margin-top: 0;
        }}
        .jawa-button {{
            background: linear-gradient(90deg, {HEADER_BG}, #1a1a1a);
            color: {CARD_BG};
            border: 1px solid {ACCENT};
            border-radius: 30px;
            padding: 8px 20px;
            font-weight: bold;
            transition: 0.3s;
        }}
        .jawa-button:hover {{
            background: linear-gradient(90deg, {ACCENT}, {ACCENT_LIGHT});
            color: #0a0a0a;
            border-color: {CARD_BG};
            transform: scale(1.02);
        }}
        .jawa-table {{
            width: 100%;
            border-collapse: collapse;
            font-size: 13px;
            background: #1e1e1e;
            color: #e6e6e6;
        }}
        .jawa-table th {{
            background: {HEADER_BG};
            color: {HEADER_TEXT};
            padding: 12px;
            font-weight: 600;
            border-bottom: 2px solid {ACCENT};
            white-space: nowrap;
        }}
        .jawa-table td {{
            padding: 10px;
            border-bottom: 1px solid #444;
        }}
        .jawa-progress {{
            border: 1px solid {ACCENT};
            border-radius: 30px;
            background: #1a1a1a;
            overflow: hidden;
        }}
        .jawa-progress-bar {{
            background: linear-gradient(90deg, #E6A341, #FECE79) !important;
            height: 30px;
            border-radius: 30px;
            text-align: center;
            line-height: 30px;
            color: black;
            font-weight: bold;
            transition: width 0.5s;
        }}
        .jawa-badge-medium {{
            background: #FECE79 !important;
            color: #1a1a1a !important;
            padding: 2px 8px;
            border-radius: 20px;
            font-size: 12px;
        }}
        .jawa-badge-high {{
            background: #8C0902;
            color: {CARD_BG};
            padding: 2px 8px;
            border-radius: 20px;
            font-size: 12px;
        }}
        .jawa-badge-low {{
            background: {HEADER_BG};
            color: {CARD_BG};
            padding: 2px 8px;
            border-radius: 20px;
            font-size: 12px;
        }}
        /* Skor bar */
        .score-bar-wrap {{
            background: #333;
            border-radius: 10px;
            width: 80px;
            height: 14px;
            display: inline-block;
            vertical-align: middle;
            margin-left: 5px;
        }}
        .score-bar-fill {{
            height: 14px;
            border-radius: 10px;
        }}
        .rank-badge {{
            display: inline-block;
            width: 26px;
            height: 26px;
            border-radius: 50%;
            text-align: center;
            line-height: 26px;
            font-weight: bold;
            font-size: 13px;
        }}
        .rank-1 {{ background: #FFD700; color: #1a1a1a; }}
        .rank-2 {{ background: #C0C0C0; color: #1a1a1a; }}
        .rank-3 {{ background: #CD7F32; color: #fff; }}
        .rank-other {{ background: #3C507D; color: #fff; }}
        .blink-text {{
            animation: blinkGold 0.8s infinite alternate;
            font-weight: bold;
        }}
        @keyframes blinkGold {{
            0% {{ color: {CARD_BG}; }}
            100% {{ color: {ACCENT_LIGHT}; text-shadow: 0 0 10px {ACCENT_LIGHT}; }}
        }}
        @keyframes bounce {{
            from {{ transform: translateY(0px); }}
            to {{ transform: translateY(-15px); }}
        }}
        .jawa-card label, .jawa-card .widget-label {{ color: {CARD_TEXT} !important; }}
        .jawa-card .widget-text input,
        .jawa-card .widget-dropdown select,
        .jawa-card .widget-inttext input {{
            background: #fff7e8;
            color: #2c2c2c;
        }}
        .formula-box {{
            background: #1a1a1a;
            border: 1px solid {ACCENT};
            border-radius: 12px;
            padding: 14px 18px;
            font-family: monospace;
            font-size: 13px;
            color: {HEADER_TEXT};
            margin: 10px 0;
            line-height: 1.8;
        }}
        .star-display {{
            color: #FFD700;
            font-size: 16px;
            letter-spacing: 2px;
        }}
        .diff-badge {{
            padding: 2px 8px;
            border-radius: 20px;
            font-size: 12px;
            font-weight: bold;
        }}
    </style>
    """

# ─── HELPER VISUAL ────────────────────────────────────────────────────────────
def bintang(n):
    return "★" * n + "☆" * (5 - n)

def score_color(skor):
    if skor >= 65: return "#c0392b"
    if skor >= 35: return "#e6a341"
    return "#27ae60"

def diff_color(d):
    colors = {1: "#27ae60", 2: "#2ecc71", 3: "#e6a341", 4: "#e67e22", 5: "#c0392b"}
    return colors.get(d, "#888")

# ─── TABEL ────────────────────────────────────────────────────────────────────
def tampilkan_tabel():
    df = get_all_tasks_df()
    if df.empty:
        return HTML("<p class='blink-text'>📭 Belum ada tugas. Silakan tambah!</p>")

    # Hitung skor tiap baris
    df['skor'] = df.apply(
        lambda r: hitung_skor(r['difficulty'] if r['difficulty'] else 3, r['deadline'])[0], axis=1)
    df['priority'] = df.apply(
        lambda r: hitung_skor(r['difficulty'] if r['difficulty'] else 3, r['deadline'])[1], axis=1)

    # Rank hanya untuk pending, diurutkan dari skor tertinggi
    pending_df = df[df['status'] == 'pending'].copy()
    pending_df = pending_df.sort_values('skor', ascending=False).reset_index(drop=True)
    pending_df['rank'] = range(1, len(pending_df) + 1)

    completed_df = df[df['status'] == 'completed'].copy()
    completed_df['rank'] = None

    df_sorted = pd.concat([pending_df, completed_df])

    html = "<table class='jawa-table'>"
    html += "<tr>"
    for col in ['Rank', 'ID', 'Nama Tugas', 'Menit', 'Deadline', 'Kesulitan', 'Skor', 'Prioritas', 'Kategori', 'Status']:
        html += f"<th>{col}</th>"
    html += "</tr>"

    for _, row in df_sorted.iterrows():
        is_done = row['status'] == 'completed'
        row_style = "style='opacity:0.5'" if is_done else ""
        skor = row['skor']
        diff = row['difficulty'] if row['difficulty'] else 3
        prio = row['priority']

        # Rank badge
        if pd.isna(row['rank']) or is_done:
            rank_html = "<td style='text-align:center'>—</td>"
        else:
            r = int(row['rank'])
            cls = {1: 'rank-1', 2: 'rank-2', 3: 'rank-3'}.get(r, 'rank-other')
            rank_html = f"<td style='text-align:center'><span class='rank-badge {cls}'>{r}</span></td>"

        # Priority badge
        if prio == 'High':
            prio_html = "<span class='jawa-badge-high'>🔴 High</span>"
            bg = "background-color:#2c1a1a"
        elif prio == 'Medium':
            prio_html = "<span class='jawa-badge-medium'>🟡 Medium</span>"
            bg = "background-color:#2a2418"
        else:
            prio_html = "<span class='jawa-badge-low'>🟢 Low</span>"
            bg = "background-color:#1a2a1a"

        if is_done:
            bg = "background-color:#1a1a1a"

        # Skor bar
        bar_color = score_color(skor)
        skor_html = f"""
            <span style='font-weight:bold;color:{bar_color}'>{skor}</span>
            <span class='score-bar-wrap'>
                <span class='score-bar-fill' style='width:{skor}%;background:{bar_color}'></span>
            </span>"""

        # Difficulty stars
        diff_html = f"<span class='star-display' style='color:{diff_color(diff)}'>{bintang(diff)}</span>"

        status_html = '✅ Selesai' if is_done else '⏳ Pending'

        html += f"<tr style='{bg}' {row_style}>"
        html += rank_html
        html += f"<td>{int(row['id'])}</td>"
        html += f"<td><b>{row['name']}</b></td>"
        html += f"<td>{row['estimated_minutes']}</td>"
        html += f"<td>{row['deadline'] or '—'}</td>"
        html += f"<td>{diff_html}</td>"
        html += f"<td>{skor_html}</td>"
        html += f"<td>{prio_html}</td>"
        html += f"<td>{row['category']}</td>"
        html += f"<td>{status_html}</td>"
        html += "</tr>"

    html += "</table>"
    return HTML(get_css() + html)

# ─── FORMULA INFO ─────────────────────────────────────────────────────────────
def tampil_formula():
    html = f"""
    <div class='formula-box'>
        <b style='color:{HEADER_TEXT};font-size:15px'>⚙️ Rumus Penghitungan Skor Prioritas</b><br><br>
        <span style='color:#aaa'>Skor (0–100) = Skor Kesulitan + Skor Urgensi Deadline</span><br><br>
        <b>Skor Kesulitan</b> = (tingkat_kesulitan / 5) × 50 &nbsp;→ maks 50 poin<br>
        &nbsp;&nbsp;&nbsp;⭐ Kesulitan 1 → +10 | ⭐⭐ 2 → +20 | ⭐⭐⭐ 3 → +30 | ⭐⭐⭐⭐ 4 → +40 | ⭐⭐⭐⭐⭐ 5 → +50<br><br>
        <b>Skor Urgensi Deadline</b>:<br>
        &nbsp;&nbsp;&nbsp;📛 Sudah lewat / hari ini → +50<br>
        &nbsp;&nbsp;&nbsp;🔴 1–3 hari lagi → +40<br>
        &nbsp;&nbsp;&nbsp;🟠 4–7 hari lagi → +25<br>
        &nbsp;&nbsp;&nbsp;🟡 8–14 hari lagi → +15<br>
        &nbsp;&nbsp;&nbsp;🟢 &gt; 14 hari lagi → +5<br><br>
        <b>Klasifikasi Prioritas Otomatis:</b><br>
        &nbsp;&nbsp;&nbsp;<span style='color:#c0392b'>🔴 High</span> = Skor ≥ 65 &nbsp;|&nbsp;
        <span style='color:#e6a341'>🟡 Medium</span> = Skor 35–64 &nbsp;|&nbsp;
        <span style='color:#27ae60'>🟢 Low</span> = Skor &lt; 35
    </div>
    """
    return HTML(get_css() + html)

# ─── PROGRESS ─────────────────────────────────────────────────────────────────
def tampil_progress_html():
    df = get_all_tasks_df()
    if df.empty:
        return HTML("<p>Belum ada tugas.</p>")
    total = len(df)
    selesai = len(df[df['status'] == 'completed'])
    pending = total - selesai
    percent = (selesai / total) * 100 if total > 0 else 0

    html = f"""
    <div class='jawa-progress'>
        <div class='jawa-progress-bar' style='width:{percent}%;'>
            {percent:.1f}%
        </div>
    </div>
    <p style='margin-top:10px'>✨ Selesai: {selesai} | Pending: {pending} | Total: {total}</p>
    """
    if percent == 100:
        html += """
        <div style="text-align:center; font-size:40px; animation: bounce 0.5s infinite alternate;">
            😊🌾🎉✨🥳
        </div>
        <div class='blink-text' style='text-align:center; font-size:24px;'>
            🌟 SELAMAT! SEMUA TUGAS ANDA SELESAI! 🌟
        </div>
        <script>
        (function() {
            if (typeof canvasConfetti === 'undefined') {
                var script = document.createElement('script');
                script.src = 'https://cdn.jsdelivr.net/npm/canvas-confetti@1';
                script.onload = function() {
                    canvasConfetti({particleCount: 300, spread: 100, origin: {y: 0.6}, colors: ['#E6A341', '#FECE79']});
                };
                document.head.appendChild(script);
            } else {
                canvasConfetti({particleCount: 300, spread: 100, origin: {y: 0.6}, colors: ['#E6A341', '#FECE79']});
            }
        })();
        </script>
        """
    return HTML(get_css() + html)

# ─── JADWAL ───────────────────────────────────────────────────────────────────
def buat_jadwal():
    df = get_all_tasks_df()
    pending = df[df['status'] == 'pending'].copy()
    if pending.empty:
        return HTML("<p>🎉 Semua tugas selesai. Tidak ada jadwal hari ini.</p>")

    # Urutkan berdasarkan skor prioritas (tertinggi duluan)
    pending['skor'] = pending.apply(
        lambda r: hitung_skor(r['difficulty'] if r['difficulty'] else 3, r['deadline'])[0], axis=1)
    pending = pending.sort_values('skor', ascending=False)

    total_menit = 8 * 60
    used = 0
    mulai = datetime.strptime("09:00", "%H:%M")
    html = f"<h3 style='color:{HEADER_BG}'>📅 Jadwal Hari Ini (8 jam, mulai 09:00) — diurutkan by Skor</h3>"
    html += "<ul style='list-style:none; padding-left:0'>"
    for _, row in pending.iterrows():
        dur = row['estimated_minutes']
        skor = row['skor']
        warna = "#8C0902" if skor >= 65 else "#E6A341" if skor >= 35 else "#210100"
        if used + dur <= total_menit:
            selesai = mulai + timedelta(minutes=dur)
            html += f"""<li style='background:{warna}; color:white; margin:8px; padding:10px;
                         border-radius:15px; border-left:5px solid {CARD_BG}'>
                         <b>{mulai.strftime('%H:%M')} – {selesai.strftime('%H:%M')}</b>
                         &nbsp;|&nbsp; {row['name']} ({dur} mnt)
                         &nbsp;|&nbsp; Skor: <b>{skor}</b>
                         &nbsp;|&nbsp; {bintang(row['difficulty'] if row['difficulty'] else 3)}
                         </li>"""
            mulai = selesai
            used += dur
        else:
            html += f"<li style='background:#333; color:#ccc; margin:8px; padding:10px; border-radius:15px'>⚠️ {row['name']} tidak muat hari ini → jadwalkan besok</li>"
    html += f"</ul><p>⏱ Total terpakai: {used/60:.1f} jam dari 8 jam</p>"
    return HTML(get_css() + html)

# ─── PLOT KATEGORI ────────────────────────────────────────────────────────────
# Warna per kategori (konsisten)
KATEGORI_COLORS = {
    "Akademik":     "#3C507D",
    "Non Akademik": "#E6A341",
    "Pribadi":      "#27ae60",
    "Organisasi":   "#8C0902",
}

def fig_to_base64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight',
                facecolor='#1a1a1a', transparent=False, dpi=110)
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode()
    plt.close(fig)
    return b64

def buat_plot_kategori():
    df = get_all_tasks_df()
    if df.empty:
        return HTML("<p style='color:#aaa'>Belum ada data.</p>")

    # Hitung skor tiap baris
    df['skor'] = df.apply(
        lambda r: hitung_skor(r['difficulty'] if r['difficulty'] else 3, r['deadline'])[0], axis=1)

    # Normalisasi nama kategori agar konsisten
    def norm_kat(k):
        if not k:
            return "Lainnya"
        kl = k.strip().lower()
        if kl in ("non akademik", "non-akademik"):
            return "Non Akademik"
        for std in KATEGORI_LIST:
            if kl == std.lower():
                return std
        return k.strip().title()

    df['kat_norm'] = df['category'].apply(norm_kat)

    # Semua kategori yang ada (supaya legend lengkap)
    all_kats = df['kat_norm'].unique().tolist()
    colors_map = {k: KATEGORI_COLORS.get(k, "#888888") for k in all_kats}

    pending_df   = df[df['status'] == 'pending']
    completed_df = df[df['status'] == 'completed']

    # ── PLOT 1: Pie chart jumlah tugas per kategori (semua) ──────────────────
    kat_count = df.groupby('kat_norm').size()
    fig1, ax1 = plt.subplots(figsize=(5, 4.5), facecolor='#1a1a1a')
    wedge_colors = [colors_map.get(k, "#888") for k in kat_count.index]
    wedges, texts, autotexts = ax1.pie(
        kat_count.values,
        labels=kat_count.index,
        autopct='%1.0f%%',
        startangle=140,
        colors=wedge_colors,
        pctdistance=0.75,
        wedgeprops=dict(linewidth=2, edgecolor='#1a1a1a')
    )
    for t in texts:
        t.set_color('#F5F0E9')
        t.set_fontsize(10)
    for at in autotexts:
        at.set_color('white')
        at.set_fontsize(9)
        at.set_fontweight('bold')
    ax1.set_title('Distribusi Tugas\nper Kategori', color='#FECE79',
                  fontsize=12, fontweight='bold', pad=10)
    ax1.set_facecolor('#1a1a1a')
    b64_1 = fig_to_base64(fig1)

    # ── PLOT 2: Bar chart rata-rata skor per kategori ─────────────────────────
    skor_by_kat = df.groupby('kat_norm')['skor'].mean().sort_values(ascending=True)
    fig2, ax2 = plt.subplots(figsize=(5, 4.5), facecolor='#1a1a1a')
    ax2.set_facecolor('#1a1a1a')
    bar_cols = [colors_map.get(k, "#888") for k in skor_by_kat.index]
    bars = ax2.barh(skor_by_kat.index, skor_by_kat.values,
                    color=bar_cols, edgecolor='#1a1a1a', linewidth=1.5, height=0.55)
    for bar, val in zip(bars, skor_by_kat.values):
        ax2.text(val + 1, bar.get_y() + bar.get_height()/2,
                 f'{val:.0f}', va='center', color='#F5F0E9', fontsize=10, fontweight='bold')
    ax2.set_xlim(0, 105)
    ax2.set_title('Rata-rata Skor Prioritas\nper Kategori', color='#FECE79',
                  fontsize=12, fontweight='bold', pad=10)
    ax2.tick_params(colors='#aaa', labelsize=10)
    ax2.spines[:].set_color('#333')
    ax2.set_xlabel('Skor (0–100)', color='#aaa', fontsize=9)
    ax2.axvline(65, color='#c0392b', linestyle='--', linewidth=1, alpha=0.6, label='High ≥65')
    ax2.axvline(35, color='#e6a341', linestyle='--', linewidth=1, alpha=0.6, label='Medium ≥35')
    ax2.legend(fontsize=8, facecolor='#1a1a1a', labelcolor='#aaa', edgecolor='#333')
    b64_2 = fig_to_base64(fig2)

    # ── PLOT 3: Stacked bar pending vs selesai per kategori ───────────────────
    kat_status = df.groupby(['kat_norm', 'status']).size().unstack(fill_value=0)
    if 'pending' not in kat_status.columns:
        kat_status['pending'] = 0
    if 'completed' not in kat_status.columns:
        kat_status['completed'] = 0

    fig3, ax3 = plt.subplots(figsize=(5, 4.5), facecolor='#1a1a1a')
    ax3.set_facecolor('#1a1a1a')
    x = range(len(kat_status.index))
    bar_w = 0.5
    b_colors = [colors_map.get(k, "#888") for k in kat_status.index]

    bars_p = ax3.bar(x, kat_status['pending'], width=bar_w,
                     color=b_colors, alpha=0.9, label='⏳ Pending', edgecolor='#1a1a1a')
    bars_c = ax3.bar(x, kat_status['completed'], width=bar_w,
                     bottom=kat_status['pending'],
                     color='#27ae60', alpha=0.6, label='✅ Selesai', edgecolor='#1a1a1a')

    ax3.set_xticks(list(x))
    ax3.set_xticklabels(kat_status.index, rotation=15, ha='right', color='#aaa', fontsize=9)
    ax3.set_title('Jumlah Tugas per Kategori\n(Pending vs Selesai)', color='#FECE79',
                  fontsize=12, fontweight='bold', pad=10)
    ax3.tick_params(colors='#aaa')
    ax3.spines[:].set_color('#333')
    ax3.set_ylabel('Jumlah Tugas', color='#aaa', fontsize=9)
    ax3.legend(fontsize=9, facecolor='#1a1a1a', labelcolor='#F5F0E9', edgecolor='#333')
    b64_3 = fig_to_base64(fig3)

    # ── HTML output ───────────────────────────────────────────────────────────
    html = f"""
    <div style='display:flex; flex-wrap:wrap; gap:16px; justify-content:center; margin-top:10px;'>
        <div style='background:#111; border:1px solid {ACCENT}; border-radius:14px; padding:12px; text-align:center;'>
            <img src='data:image/png;base64,{b64_1}' style='max-width:280px; border-radius:8px'/>
        </div>
        <div style='background:#111; border:1px solid {ACCENT}; border-radius:14px; padding:12px; text-align:center;'>
            <img src='data:image/png;base64,{b64_2}' style='max-width:280px; border-radius:8px'/>
        </div>
        <div style='background:#111; border:1px solid {ACCENT}; border-radius:14px; padding:12px; text-align:center;'>
            <img src='data:image/png;base64,{b64_3}' style='max-width:280px; border-radius:8px'/>
        </div>
    </div>
    """
    return HTML(get_css() + html)
nama_input      = widgets.Text(placeholder="Nama tugas", description="📝 Nama:",
                               layout=widgets.Layout(width='500px'))
waktu_input     = widgets.IntSlider(min=5, max=500, step=5, value=30, description="⏱ Menit:",
                                    layout=widgets.Layout(width='500px'))
deadline_input  = widgets.DatePicker(description="📅 Deadline:",
                                     layout=widgets.Layout(width='500px'))
kategori_input  = widgets.Dropdown(
    options=KATEGORI_LIST,
    value="Akademik",
    description="📂 Kategori:",
    layout=widgets.Layout(width='500px')
)

# ★ Difficulty rating baru (1–5)
difficulty_input = widgets.RadioButtons(
    options=[(f'{bintang(i)}  ({i})', i) for i in range(1, 6)],
    value=3,
    description="🎯 Kesulitan:",
    layout=widgets.Layout(width='500px')
)

# Preview skor live
out_score_preview = widgets.Output()

def update_score_preview(*args):
    dl = deadline_input.value.isoformat() if deadline_input.value else None
    skor, prio = hitung_skor(difficulty_input.value, dl)
    bar_color = score_color(skor)
    prio_color = {"High": "#c0392b", "Medium": "#e6a341", "Low": "#27ae60"}[prio]
    with out_score_preview:
        clear_output(wait=True)
        display(HTML(get_css() + f"""
        <div style='background:#1a1a1a; border:1px solid {ACCENT}; border-radius:12px;
                    padding:12px 18px; display:inline-block; margin-top:5px;'>
            <span style='color:#aaa; font-size:13px'>Estimasi Skor: </span>
            <span style='font-size:22px; font-weight:bold; color:{bar_color}'>{skor}</span>
            <span class='score-bar-wrap' style='width:120px; vertical-align:middle; margin:0 10px'>
                <span class='score-bar-fill' style='width:{skor}%; background:{bar_color}'></span>
            </span>
            <span style='color:{prio_color}; font-weight:bold'>→ {prio}</span>
        </div>
        """))

difficulty_input.observe(update_score_preview, names='value')
deadline_input.observe(update_score_preview, names='value')

btn_tambah = widgets.Button(description="➕ TAMBAH TUGAS", layout=widgets.Layout(width='300px'))
btn_reset  = widgets.Button(description="🗑 HAPUS SEMUA",  layout=widgets.Layout(width='300px'))
btn_tambah.add_class('jawa-button')
btn_reset.add_class('jawa-button')

out_tabel   = widgets.Output()
out_progress = widgets.Output()
out_jadwal  = widgets.Output()
out_status  = widgets.Output()
out_formula = widgets.Output()
out_plot_kat = widgets.Output()

def refresh_all():
    recalc_all_scores()
    with out_tabel:
        clear_output(wait=True)
        display(HTML(get_css()))
        display(tampilkan_tabel())
    with out_progress:
        clear_output(wait=True)
        display(HTML(get_css()))
        display(tampil_progress_html())
    with out_jadwal:
        clear_output(wait=True)
        display(HTML(get_css()))
        display(buat_jadwal())
    with out_formula:
        clear_output(wait=True)
        display(tampil_formula())
    with out_plot_kat:
        clear_output(wait=True)
        display(HTML(get_css()))
        display(buat_plot_kategori())

def tambah(btn):
    with out_status:
        clear_output()
        nama = nama_input.value.strip()
        if not nama:
            print("⚠️ Nama tugas harus diisi!")
            return
        dl = deadline_input.value.isoformat() if deadline_input.value else None
        add_task(nama, waktu_input.value, dl, kategori_input.value, difficulty_input.value)
        skor, prio = hitung_skor(difficulty_input.value, dl)
        print(f"✅ '{nama}' ditambahkan! Skor: {skor} → Prioritas: {prio}")
        nama_input.value = ""
        waktu_input.value = 30
        deadline_input.value = None
        kategori_input.value = "Akademik"
        difficulty_input.value = 3
        refresh_all()
        update_score_preview()

def hapus_semua(btn):
    with out_status:
        clear_output()
        clear_all_tasks()
        print("🗑 Semua tugas telah dihapus.")
        refresh_all()
        insert_sample_tasks()
        refresh_all()

btn_tambah.on_click(tambah)
btn_reset.on_click(hapus_semua)

# Tandai selesai
id_selesai  = widgets.IntText(description="ID Tugas:", value=1, layout=widgets.Layout(width='150px'))
btn_selesai = widgets.Button(description="✔️ TANDAI SELESAI", layout=widgets.Layout(width='180px'))
btn_selesai.add_class('jawa-button')

def selesaikan(btn):
    with out_status:
        clear_output()
        tid = id_selesai.value
        sukses, pesan = complete_task_by_id(tid)
        if sukses:
            print(f"✅ Tugas ID {tid} telah diselesaikan!")
            refresh_all()
        else:
            print(f"❌ Gagal: {pesan}. Cek ID {tid} dan pastikan status masih Pending.")
        id_selesai.value = 1

btn_selesai.on_click(selesaikan)

# ─── TAMPILAN UTAMA ───────────────────────────────────────────────────────────
display(HTML(get_css()))
display(HTML(f"""
<div class='jawa-container'>
    <div class='jawa-title'>ChronoScore</div>
    <div style='text-align:center;'><span class='jawa-sub'>TO DO LIST BY KELOMPOK 6</span></div>
"""))

# 1. Rumus
display(HTML("<div class='jawa-card'><h3>⚙️ SISTEM SCORING PRIORITAS</h3>"))
display(out_formula)
display(HTML("</div>"))

# 2. Form tambah
display(HTML("<div class='jawa-card'><h3 style='text-align:center'>✍️ TAMBAH TUGAS BARU</h3>"
             "<div style='display:flex; justify-content:center;'>"))
display(widgets.VBox([
    nama_input, waktu_input, deadline_input, kategori_input,
    difficulty_input,
    out_score_preview,
    widgets.HBox([btn_tambah, btn_reset])
], layout=widgets.Layout(align_items='center')))
display(HTML("</div></div>"))

# 3. Tabel
display(HTML("<div class='jawa-card'><h3>📋 DAFTAR TUGAS (diurutkan by Skor)</h3>"))
display(out_tabel)
display(HTML("</div>"))

# 4. Tandai selesai
display(HTML("<div class='jawa-card'><h3>✅ TANDAI TUGAS SELESAI</h3>"))
display(widgets.HBox([id_selesai, btn_selesai]))
display(HTML("</div>"))

# 5. Progress & Jadwal
display(HTML("<div class='jawa-card'><h3>📊 PROGRESS & JADWAL</h3>"))
display(out_progress)
display(out_jadwal)
display(HTML("</div>"))

# 6. Plot kategori
display(HTML("<div class='jawa-card'><h3>📈 ANALISIS PER KATEGORI</h3>"))
display(out_plot_kat)
display(HTML("</div>"))

display(HTML("</div>"))
display(out_status)

refresh_all()
update_score_preview()

Output()

Output()

Output()

Output()

Output()

Output()

In [6]:
import sqlite3
from datetime import datetime, timedelta, date
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import io, base64

# ─── KATEGORI (hanya 2) ───────────────────────────────────────────────────────
KATEGORI_LIST = ["Akademik", "Non Akademik"]

# ─── PALET WARNA (dari Color Palette 101) ────────────────────────────────────
FAIRY_LIGHTS      = "#FFDDBD"   # krem peach — bg card, teks terang
BLUE_TOURMALIN    = "#5FBBEF"   # biru langit — aksen utama
BLUE_HEPATICA     = "#5C71DD"   # biru medium — aksen sekunder
CANDIED_BLUEBERRY = "#522196"   # ungu gelap — header bg
BANAISAGI_PURPLE  = "#CD2382"   # magenta/pink tua — danger/high
PINK_KATYDID      = "#FF62BB"   # pink cerah — aksen warm

# shorthand
BG_MAIN   = "#120820"           # latar gelap nuansa ungu
CARD_BG   = CANDIED_BLUEBERRY   # card bg: ungu gelap
CARD_TEXT = FAIRY_LIGHTS        # teks di card: krem
ACCENT    = BLUE_TOURMALIN      # border/aksen: biru langit
ACCENT2   = BLUE_HEPATICA       # aksen sekunder: biru medium
HEADER_BG = "#1a0535"           # header sangat gelap
HEADER_TXT= FAIRY_LIGHTS        # teks header: krem
TITLE_CLR = PINK_KATYDID        # warna judul
SUB_CLR   = BLUE_TOURMALIN

# Warna per kategori di chart
KATEGORI_COLORS = {
    "Akademik":     BLUE_TOURMALIN,
    "Non Akademik": PINK_KATYDID,
}

# ─── DATABASE ─────────────────────────────────────────────────────────────────
DB_NAME = "tasks.db"

def init_db():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute('''CREATE TABLE IF NOT EXISTS tasks (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        estimated_minutes INTEGER,
        deadline TEXT,
        priority TEXT,
        category TEXT,
        difficulty INTEGER DEFAULT 3,
        status TEXT DEFAULT 'pending',
        created_at TEXT DEFAULT CURRENT_TIMESTAMP,
        completed_at TEXT
    )''')
    try:
        c.execute("ALTER TABLE tasks ADD COLUMN difficulty INTEGER DEFAULT 3")
    except:
        pass
    conn.commit()
    conn.close()

init_db()

# ─── RUMUS SKOR ───────────────────────────────────────────────────────────────
def hitung_skor(difficulty, deadline_str):
    difficulty_score = (difficulty / 5) * 50
    if deadline_str:
        try:
            dl = datetime.strptime(deadline_str, "%Y-%m-%d").date()
            hari_tersisa = (dl - date.today()).days
            if hari_tersisa <= 0:   urgency_score = 50
            elif hari_tersisa <= 3: urgency_score = 40
            elif hari_tersisa <= 7: urgency_score = 25
            elif hari_tersisa <= 14:urgency_score = 15
            else:                   urgency_score = 5
        except:
            urgency_score = 0
    else:
        urgency_score = 0
    skor = round(difficulty_score + urgency_score)
    priority = "High" if skor >= 65 else "Medium" if skor >= 35 else "Low"
    return skor, priority

# ─── FUNGSI DB ────────────────────────────────────────────────────────────────
def add_task(name, minutes, deadline, category, difficulty):
    skor, priority = hitung_skor(difficulty, deadline)
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute('''INSERT INTO tasks (name, estimated_minutes, deadline, priority, category, difficulty, status)
                 VALUES (?,?,?,?,?,?,'pending')''',
              (name, minutes, deadline, priority, category, difficulty))
    conn.commit()
    conn.close()

def get_all_tasks_df():
    conn = sqlite3.connect(DB_NAME)
    df = pd.read_sql_query(
        "SELECT id, name, estimated_minutes, deadline, priority, category, difficulty, status FROM tasks", conn)
    conn.close()
    return df

def clear_all_tasks():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("DELETE FROM tasks")
    conn.commit()
    conn.close()

def complete_task_by_id(task_id):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    try:
        c.execute("SELECT status FROM tasks WHERE id=?", (task_id,))
        row = c.fetchone()
        if row is None:       return False, "ID tidak ditemukan"
        if row[0] != 'pending': return False, "Tugas sudah selesai"
        now = datetime.now().isoformat()
        c.execute("UPDATE tasks SET status='completed', completed_at=? WHERE id=? AND status='pending'",
                  (now, task_id))
        conn.commit()
        return c.rowcount > 0, "Berhasil"
    except Exception as e:
        return False, str(e)
    finally:
        conn.close()

def recalc_all_scores():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT id, difficulty, deadline FROM tasks WHERE status='pending'")
    for tid, diff, dl in c.fetchall():
        skor, prio = hitung_skor(diff or 3, dl)
        c.execute("UPDATE tasks SET priority=? WHERE id=?", (prio, tid))
    conn.commit()
    conn.close()

# ─── DATA CONTOH ──────────────────────────────────────────────────────────────
def insert_sample_tasks():
    if get_all_tasks_df().empty:
        sample = [
            ("Belajar UAP",      100, "2026-06-12", "Akademik",     4),
            ("Menulis Ilmiah",    60, "2026-06-10", "Akademik",     3),
            ("Video Resume FPO",  90, "2026-06-13", "Non Akademik", 3),
            ("Notulensi",         30, "2026-06-09", "Non Akademik", 2),
            ("Presentasi Akhir",  75, "2026-06-14", "Akademik",     5),
        ]
        for nama, menit, deadline, kat, diff in sample:
            add_task(nama, menit, deadline, kat, diff)
        print("✅ Data contoh ditambahkan.\n")

insert_sample_tasks()
recalc_all_scores()

# ─── CSS ──────────────────────────────────────────────────────────────────────
def get_css():
    return f"""
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@400;700&family=Poppins:wght@300;400;600&display=swap');

        .cs-container {{
            font-family: 'Poppins', sans-serif;
            background: linear-gradient(135deg, {BG_MAIN} 0%, #1e0a3c 60%, #2a0a2e 100%);
            color: {FAIRY_LIGHTS};
            padding: 28px;
            border-radius: 24px;
            box-shadow: 0 0 40px rgba(205,35,130,0.25), 0 0 80px rgba(82,33,150,0.2);
            border: 1.5px solid {BLUE_HEPATICA};
        }}
        .cs-title {{
            font-family: 'Playfair Display', serif;
            font-size: 5em;
            font-weight: 800;
            text-align: center;
            background: linear-gradient(90deg, {BLUE_TOURMALIN}, {PINK_KATYDID}, {BLUE_HEPATICA});
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
            background-clip: text;
            text-shadow: none;
            letter-spacing: 3px;
            margin-bottom: 8px;
            filter: drop-shadow(0 0 18px {BANAISAGI_PURPLE});
        }}
        .cs-sub {{
            text-align: center;
            font-style: italic;
            color: {BLUE_TOURMALIN};
            margin-bottom: 20px;
            border-top: 1px solid {BLUE_HEPATICA};
            border-bottom: 1px solid {BLUE_HEPATICA};
            display: inline-block;
            padding: 5px 24px;
            letter-spacing: 2px;
            font-size: 0.85em;
        }}
        .cs-card {{
            background: linear-gradient(135deg, rgba(82,33,150,0.55) 0%, rgba(26,5,53,0.85) 100%);
            border-radius: 18px;
            padding: 18px 20px;
            margin: 14px 0;
            border: 1px solid {BLUE_HEPATICA};
            box-shadow: 0 4px 24px rgba(205,35,130,0.12), inset 0 1px 0 rgba(255,221,189,0.08);
            color: {FAIRY_LIGHTS};
            backdrop-filter: blur(4px);
        }}
        .cs-card h3 {{
            color: {PINK_KATYDID};
            font-weight: 700;
            margin-top: 0;
            letter-spacing: 1px;
            text-transform: uppercase;
            font-size: 0.95em;
        }}
        .cs-button {{
            background: linear-gradient(90deg, {CANDIED_BLUEBERRY}, {BANAISAGI_PURPLE});
            color: {FAIRY_LIGHTS};
            border: 1px solid {PINK_KATYDID};
            border-radius: 30px;
            padding: 9px 22px;
            font-weight: 700;
            font-size: 13px;
            letter-spacing: 0.5px;
            cursor: pointer;
            transition: all 0.25s;
            box-shadow: 0 0 12px rgba(255,98,187,0.3);
        }}
        .cs-button:hover {{
            background: linear-gradient(90deg, {BANAISAGI_PURPLE}, {PINK_KATYDID});
            box-shadow: 0 0 20px rgba(255,98,187,0.6);
            transform: scale(1.03);
        }}
        .cs-table {{
            width: 100%;
            border-collapse: collapse;
            font-size: 13px;
            background: rgba(18,8,32,0.8);
            color: {FAIRY_LIGHTS};
        }}
        .cs-table th {{
            background: linear-gradient(90deg, {CANDIED_BLUEBERRY}, #3a1070);
            color: {FAIRY_LIGHTS};
            padding: 12px 10px;
            font-weight: 600;
            border-bottom: 2px solid {PINK_KATYDID};
            white-space: nowrap;
            font-size: 12px;
            letter-spacing: 0.5px;
        }}
        .cs-table td {{
            padding: 10px;
            border-bottom: 1px solid rgba(92,113,221,0.25);
        }}
        .cs-progress {{
            border: 1px solid {BLUE_HEPATICA};
            border-radius: 30px;
            background: rgba(18,8,32,0.8);
            overflow: hidden;
        }}
        .cs-progress-bar {{
            background: linear-gradient(90deg, {BANAISAGI_PURPLE}, {PINK_KATYDID}, {BLUE_TOURMALIN}) !important;
            height: 30px;
            border-radius: 30px;
            text-align: center;
            line-height: 30px;
            color: white;
            font-weight: bold;
            transition: width 0.6s;
            text-shadow: 0 1px 3px rgba(0,0,0,0.5);
        }}
        /* Priority badges */
        .badge-high   {{ background: {BANAISAGI_PURPLE}; color: white; padding: 3px 10px; border-radius: 20px; font-size: 12px; font-weight:700; box-shadow:0 0 8px rgba(205,35,130,0.5); }}
        .badge-medium {{ background: {BLUE_HEPATICA};    color: white; padding: 3px 10px; border-radius: 20px; font-size: 12px; font-weight:700; }}
        .badge-low    {{ background: {CANDIED_BLUEBERRY};color: {FAIRY_LIGHTS}; padding: 3px 10px; border-radius: 20px; font-size: 12px; font-weight:700; }}
        /* Kategori badges */
        .badge-akademik    {{ background: {BLUE_TOURMALIN};  color: #120820; padding: 3px 10px; border-radius: 20px; font-size: 11px; font-weight:700; }}
        .badge-nonakademik {{ background: {PINK_KATYDID};    color: #120820; padding: 3px 10px; border-radius: 20px; font-size: 11px; font-weight:700; }}
        /* Score bar */
        .score-bar-wrap {{ background: rgba(255,255,255,0.1); border-radius: 10px; width:80px; height:12px; display:inline-block; vertical-align:middle; margin-left:5px; }}
        .score-bar-fill {{ height:12px; border-radius:10px; }}
        /* Rank badges */
        .rank-badge {{ display:inline-block; width:26px; height:26px; border-radius:50%; text-align:center; line-height:26px; font-weight:bold; font-size:13px; }}
        .rank-1 {{ background: linear-gradient(135deg,#FFD700,#FFA500); color:#120820; box-shadow:0 0 8px #FFD700; }}
        .rank-2 {{ background: linear-gradient(135deg,#C0C0C0,#888);    color:#120820; }}
        .rank-3 {{ background: linear-gradient(135deg,#CD7F32,#8B4513); color:white; }}
        .rank-other {{ background: {CANDIED_BLUEBERRY}; color: {FAIRY_LIGHTS}; }}
        /* Formula box */
        .formula-box {{
            background: rgba(18,8,32,0.9);
            border: 1px solid {BLUE_HEPATICA};
            border-radius: 14px;
            padding: 16px 20px;
            font-family: monospace;
            font-size: 13px;
            color: {FAIRY_LIGHTS};
            margin: 8px 0;
            line-height: 2;
        }}
        .star-gold {{ color: #FFD700; letter-spacing: 2px; font-size:15px; }}
        .blink-text {{
            animation: blinkPink 0.8s infinite alternate;
            font-weight: bold;
        }}
        @keyframes blinkPink {{
            0%   {{ color: {FAIRY_LIGHTS}; }}
            100% {{ color: {PINK_KATYDID}; text-shadow: 0 0 10px {PINK_KATYDID}; }}
        }}
        @keyframes bounce {{
            from {{ transform: translateY(0px); }}
            to   {{ transform: translateY(-15px); }}
        }}
        /* Widget label overrides inside cs-card */
        .cs-card label, .cs-card .widget-label {{ color: {FAIRY_LIGHTS} !important; }}
        .cs-card .widget-text input,
        .cs-card .widget-dropdown select,
        .cs-card .widget-inttext input {{
            background: rgba(82,33,150,0.4) !important;
            color: {FAIRY_LIGHTS} !important;
            border: 1px solid {BLUE_HEPATICA} !important;
            border-radius: 8px !important;
        }}
        .cs-card .widget-slider .slider .track-highlight {{ background: {PINK_KATYDID} !important; }}
    </style>
    """

# ─── HELPER ───────────────────────────────────────────────────────────────────
def bintang(n):
    return "★" * n + "☆" * (5 - n)

def score_color(skor):
    if skor >= 65: return BANAISAGI_PURPLE
    if skor >= 35: return BLUE_HEPATICA
    return BLUE_TOURMALIN

def norm_kat(k):
    if not k: return "Non Akademik"
    kl = k.strip().lower()
    if kl in ("non akademik","non-akademik","nonakademik"): return "Non Akademik"
    if kl == "akademik": return "Akademik"
    # mapping lama → baru
    if kl in ("pribadi","olahraga","organisasi"): return "Non Akademik"
    return "Non Akademik"

# ─── TABEL ────────────────────────────────────────────────────────────────────
def tampilkan_tabel():
    df = get_all_tasks_df()
    if df.empty:
        return HTML(f"<p class='blink-text'>📭 Belum ada tugas. Silakan tambah!</p>")

    df['skor']     = df.apply(lambda r: hitung_skor(r['difficulty'] or 3, r['deadline'])[0], axis=1)
    df['priority'] = df.apply(lambda r: hitung_skor(r['difficulty'] or 3, r['deadline'])[1], axis=1)
    df['kat_norm'] = df['category'].apply(norm_kat)

    pending_df   = df[df['status']=='pending'].copy().sort_values('skor', ascending=False).reset_index(drop=True)
    pending_df['rank'] = range(1, len(pending_df)+1)
    completed_df = df[df['status']=='completed'].copy()
    completed_df['rank'] = None
    df_sorted = pd.concat([pending_df, completed_df])

    html = "<table class='cs-table'>"
    html += "<tr>" + "".join(f"<th>{c}</th>" for c in
            ['Rank','ID','Nama Tugas','Menit','Deadline','Kesulitan','Skor','Prioritas','Kategori','Status']) + "</tr>"

    for _, row in df_sorted.iterrows():
        is_done = row['status'] == 'completed'
        skor  = row['skor']
        diff  = int(row['difficulty']) if row['difficulty'] else 3
        prio  = row['priority']
        kat   = row['kat_norm']

        # row bg
        if is_done:
            bg = "background:rgba(82,33,150,0.15); opacity:0.55"
        elif skor >= 65:
            bg = f"background:rgba(205,35,130,0.15)"
        elif skor >= 35:
            bg = f"background:rgba(92,113,221,0.15)"
        else:
            bg = f"background:rgba(95,187,239,0.1)"

        # rank badge
        if is_done or pd.isna(row['rank']):
            rank_html = "<td style='text-align:center'>—</td>"
        else:
            r = int(row['rank'])
            cls = {1:'rank-1',2:'rank-2',3:'rank-3'}.get(r,'rank-other')
            rank_html = f"<td style='text-align:center'><span class='rank-badge {cls}'>{r}</span></td>"

        prio_html = (f"<span class='badge-high'>🔴 High</span>" if prio=='High'
                else f"<span class='badge-medium'>🟡 Medium</span>" if prio=='Medium'
                else f"<span class='badge-low'>🟢 Low</span>")

        kat_cls = "badge-akademik" if kat=="Akademik" else "badge-nonakademik"
        kat_html = f"<span class='{kat_cls}'>{kat}</span>"

        sc = score_color(skor)
        skor_html = (f"<span style='font-weight:bold;color:{sc}'>{skor}</span>"
                     f"<span class='score-bar-wrap'>"
                     f"<span class='score-bar-fill' style='width:{skor}%;background:{sc}'></span></span>")

        diff_html = f"<span class='star-gold'>{bintang(diff)}</span>"
        status_html = '✅ Selesai' if is_done else '⏳ Pending'

        html += f"<tr style='{bg}'>"
        html += rank_html
        html += f"<td>{int(row['id'])}</td>"
        html += f"<td><b>{row['name']}</b></td>"
        html += f"<td>{row['estimated_minutes']}</td>"
        html += f"<td>{row['deadline'] or '—'}</td>"
        html += f"<td>{diff_html}</td>"
        html += f"<td>{skor_html}</td>"
        html += f"<td>{prio_html}</td>"
        html += f"<td>{kat_html}</td>"
        html += f"<td>{status_html}</td>"
        html += "</tr>"
    html += "</table>"
    return HTML(get_css() + html)

# ─── FORMULA ──────────────────────────────────────────────────────────────────
def tampil_formula():
    html = f"""
    <div class='formula-box'>
        <b style='color:{PINK_KATYDID};font-size:15px'>⚙️ Rumus Penghitungan Skor Prioritas</b><br><br>
        <span style='color:{BLUE_TOURMALIN}'>Skor (0–100) = Skor Kesulitan + Skor Urgensi Deadline</span><br><br>
        <b style='color:{FAIRY_LIGHTS}'>Skor Kesulitan</b> = (tingkat_kesulitan / 5) × 50 &nbsp;→ maks 50 poin<br>
        &nbsp;&nbsp;⭐ ×1→+10 &nbsp;⭐×2→+20 &nbsp;⭐×3→+30 &nbsp;⭐×4→+40 &nbsp;⭐×5→+50<br><br>
        <b style='color:{FAIRY_LIGHTS}'>Skor Urgensi Deadline</b>:<br>
        &nbsp;&nbsp;<span style='color:{PINK_KATYDID}'>● Sudah lewat / hari ini → +50</span><br>
        &nbsp;&nbsp;<span style='color:{BANAISAGI_PURPLE}'>● 1–3 hari → +40</span> &nbsp;
              <span style='color:{BLUE_HEPATICA}'>● 4–7 hari → +25</span> &nbsp;
              <span style='color:{BLUE_TOURMALIN}'>● 8–14 hari → +15</span> &nbsp;
              <span style='color:{FAIRY_LIGHTS}'>● &gt;14 hari → +5</span><br><br>
        <b>Klasifikasi:</b> &nbsp;
        <span class='badge-high'>🔴 High ≥65</span> &nbsp;
        <span class='badge-medium'>🟡 Medium 35–64</span> &nbsp;
        <span class='badge-low'>🟢 Low &lt;35</span>
    </div>
    """
    return HTML(get_css() + html)

# ─── PROGRESS ─────────────────────────────────────────────────────────────────
def tampil_progress_html():
    df = get_all_tasks_df()
    if df.empty:
        return HTML("<p>Belum ada tugas.</p>")
    total   = len(df)
    selesai = len(df[df['status']=='completed'])
    pending = total - selesai
    percent = (selesai / total) * 100 if total > 0 else 0
    html = f"""
    <div class='cs-progress'>
        <div class='cs-progress-bar' style='width:{percent}%;'>
            {percent:.1f}%
        </div>
    </div>
    <p style='margin-top:10px;color:{BLUE_TOURMALIN}'>✨ Selesai: <b>{selesai}</b> &nbsp;|&nbsp; Pending: <b>{pending}</b> &nbsp;|&nbsp; Total: <b>{total}</b></p>
    """
    if percent == 100:
        html += f"""
        <div style="text-align:center;font-size:40px;animation:bounce 0.5s infinite alternate;">😊🌸🎉✨🥳</div>
        <div class='blink-text' style='text-align:center;font-size:22px;'>🌟 SELAMAT! SEMUA TUGAS SELESAI! 🌟</div>
        <script>
        (function(){{
            if(typeof canvasConfetti==='undefined'){{
                var s=document.createElement('script');
                s.src='https://cdn.jsdelivr.net/npm/canvas-confetti@1';
                s.onload=function(){{canvasConfetti({{particleCount:300,spread:100,origin:{{y:0.6}},colors:['{PINK_KATYDID}','{BLUE_TOURMALIN}','{FAIRY_LIGHTS}']}});}};
                document.head.appendChild(s);
            }} else {{
                canvasConfetti({{particleCount:300,spread:100,origin:{{y:0.6}},colors:['{PINK_KATYDID}','{BLUE_TOURMALIN}','{FAIRY_LIGHTS}']}});
            }}
        }})();
        </script>
        """
    return HTML(get_css() + html)

# ─── JADWAL ───────────────────────────────────────────────────────────────────
def buat_jadwal():
    df = get_all_tasks_df()
    pending = df[df['status']=='pending'].copy()
    if pending.empty:
        return HTML(f"<p style='color:{BLUE_TOURMALIN}'>🎉 Semua tugas selesai. Tidak ada jadwal hari ini.</p>")
    pending['skor'] = pending.apply(
        lambda r: hitung_skor(r['difficulty'] or 3, r['deadline'])[0], axis=1)
    pending = pending.sort_values('skor', ascending=False)

    total_menit = 8*60
    used   = 0
    mulai  = datetime.strptime("09:00", "%H:%M")
    html   = f"<h3 style='color:{PINK_KATYDID}'>📅 Jadwal Hari Ini (8 jam, mulai 09:00) — diurutkan by Skor</h3>"
    html  += "<ul style='list-style:none;padding-left:0'>"
    for _, row in pending.iterrows():
        dur  = row['estimated_minutes']
        skor = row['skor']
        diff = int(row['difficulty']) if row['difficulty'] else 3
        kat  = norm_kat(row['category'])
        warna = BANAISAGI_PURPLE if skor>=65 else BLUE_HEPATICA if skor>=35 else CANDIED_BLUEBERRY
        border = PINK_KATYDID if skor>=65 else BLUE_TOURMALIN
        if used + dur <= total_menit:
            selesai_t = mulai + timedelta(minutes=dur)
            kat_badge = f"<span style='background:{'"+BLUE_TOURMALIN+"' if kat=='Akademik' else '"+PINK_KATYDID+"'};color:#120820;border-radius:12px;padding:1px 8px;font-size:11px;font-weight:bold'>{kat}</span>"
            html += (f"<li style='background:{warna};color:white;margin:8px;padding:11px 14px;"
                     f"border-radius:14px;border-left:5px solid {border}'>"
                     f"<b>{mulai.strftime('%H:%M')} – {selesai_t.strftime('%H:%M')}</b>"
                     f" &nbsp;|&nbsp; {row['name']} ({dur} mnt)"
                     f" &nbsp;|&nbsp; Skor: <b>{skor}</b>"
                     f" &nbsp;|&nbsp; <span style='color:#FFD700'>{bintang(diff)}</span>"
                     f" &nbsp;{kat_badge}</li>")
            mulai  = selesai_t
            used  += dur
        else:
            html += (f"<li style='background:rgba(255,255,255,0.07);color:#aaa;margin:8px;padding:10px;"
                     f"border-radius:14px'>⚠️ {row['name']} tidak muat hari ini → jadwalkan besok</li>")
    html += f"</ul><p style='color:{BLUE_TOURMALIN}'>⏱ Total terpakai: <b>{used/60:.1f} jam</b> dari 8 jam</p>"
    return HTML(get_css() + html)

# ─── PLOT KATEGORI ────────────────────────────────────────────────────────────
PLOT_BG   = "#120820"
PLOT_GRID = "#2a1050"

def fig_to_base64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', facecolor=PLOT_BG, dpi=110)
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode()
    plt.close(fig)
    return b64

def buat_plot_kategori():
    df = get_all_tasks_df()
    if df.empty:
        return HTML(f"<p style='color:{BLUE_TOURMALIN}'>Belum ada data.</p>")

    df['skor']     = df.apply(lambda r: hitung_skor(r['difficulty'] or 3, r['deadline'])[0], axis=1)
    df['kat_norm'] = df['category'].apply(norm_kat)
    colors_map     = KATEGORI_COLORS.copy()

    # ── PIE: distribusi jumlah tugas ─────────────────────────────────────────
    kat_count  = df.groupby('kat_norm').size().reindex(KATEGORI_LIST, fill_value=0)
    kat_count  = kat_count[kat_count > 0]
    fig1, ax1  = plt.subplots(figsize=(4.8, 4.4), facecolor=PLOT_BG)
    ax1.set_facecolor(PLOT_BG)
    pie_colors = [colors_map.get(k, "#888") for k in kat_count.index]
    wedges, texts, autotexts = ax1.pie(
        kat_count.values, labels=kat_count.index,
        autopct='%1.0f%%', startangle=140, colors=pie_colors,
        pctdistance=0.72, wedgeprops=dict(linewidth=2.5, edgecolor=PLOT_BG),
        textprops={'color': FAIRY_LIGHTS, 'fontsize': 10}
    )
    for at in autotexts:
        at.set_color('white'); at.set_fontsize(11); at.set_fontweight('bold')
    ax1.set_title('Distribusi Tugas\nper Kategori', color=PINK_KATYDID,
                  fontsize=12, fontweight='bold', pad=12)
    b64_1 = fig_to_base64(fig1)

    # ── BAR H: rata-rata skor ─────────────────────────────────────────────────
    skor_by_kat = df.groupby('kat_norm')['skor'].mean().reindex(KATEGORI_LIST).dropna()
    fig2, ax2   = plt.subplots(figsize=(4.8, 4.4), facecolor=PLOT_BG)
    ax2.set_facecolor(PLOT_BG)
    bar_colors  = [colors_map.get(k, "#888") for k in skor_by_kat.index]
    bars = ax2.barh(skor_by_kat.index, skor_by_kat.values,
                    color=bar_colors, edgecolor=PLOT_BG, linewidth=1.5, height=0.45)
    for bar, val in zip(bars, skor_by_kat.values):
        ax2.text(val + 1, bar.get_y() + bar.get_height()/2,
                 f'{val:.0f}', va='center', color=FAIRY_LIGHTS, fontsize=11, fontweight='bold')
    ax2.set_xlim(0, 108)
    ax2.set_title('Rata-rata Skor Prioritas\nper Kategori', color=PINK_KATYDID,
                  fontsize=12, fontweight='bold', pad=12)
    ax2.tick_params(colors=FAIRY_LIGHTS, labelsize=10)
    for spine in ax2.spines.values(): spine.set_color(PLOT_GRID)
    ax2.set_xlabel('Skor (0–100)', color=BLUE_TOURMALIN, fontsize=9)
    ax2.axvline(65, color=BANAISAGI_PURPLE, linestyle='--', linewidth=1.2, alpha=0.8, label='High ≥65')
    ax2.axvline(35, color=BLUE_HEPATICA,   linestyle='--', linewidth=1.2, alpha=0.8, label='Medium ≥35')
    ax2.legend(fontsize=8, facecolor=PLOT_BG, labelcolor=FAIRY_LIGHTS, edgecolor=PLOT_GRID)
    b64_2 = fig_to_base64(fig2)

    # ── STACKED BAR: pending vs selesai ──────────────────────────────────────
    kat_status = df.groupby(['kat_norm','status']).size().unstack(fill_value=0)
    if 'pending'   not in kat_status.columns: kat_status['pending']   = 0
    if 'completed' not in kat_status.columns: kat_status['completed'] = 0
    kat_status = kat_status.reindex(KATEGORI_LIST).fillna(0)

    fig3, ax3  = plt.subplots(figsize=(4.8, 4.4), facecolor=PLOT_BG)
    ax3.set_facecolor(PLOT_BG)
    x          = range(len(kat_status.index))
    b_colors   = [colors_map.get(k,"#888") for k in kat_status.index]
    ax3.bar(x, kat_status['pending'],   width=0.45, color=b_colors,    alpha=0.92, label='⏳ Pending',  edgecolor=PLOT_BG)
    ax3.bar(x, kat_status['completed'], width=0.45, bottom=kat_status['pending'],
            color='#27ae60', alpha=0.75, label='✅ Selesai', edgecolor=PLOT_BG)
    ax3.set_xticks(list(x))
    ax3.set_xticklabels(kat_status.index, color=FAIRY_LIGHTS, fontsize=10)
    ax3.set_title('Pending vs Selesai\nper Kategori', color=PINK_KATYDID,
                  fontsize=12, fontweight='bold', pad=12)
    ax3.tick_params(colors=FAIRY_LIGHTS, labelsize=10)
    for spine in ax3.spines.values(): spine.set_color(PLOT_GRID)
    ax3.set_ylabel('Jumlah Tugas', color=BLUE_TOURMALIN, fontsize=9)
    ax3.legend(fontsize=9, facecolor=PLOT_BG, labelcolor=FAIRY_LIGHTS, edgecolor=PLOT_GRID)
    b64_3 = fig_to_base64(fig3)

    html = f"""
    <div style='display:flex;flex-wrap:wrap;gap:14px;justify-content:center;margin-top:8px;'>
        <div style='background:rgba(82,33,150,0.3);border:1px solid {BLUE_HEPATICA};border-radius:14px;padding:12px;text-align:center;'>
            <img src='data:image/png;base64,{b64_1}' style='max-width:270px;border-radius:8px'/>
        </div>
        <div style='background:rgba(82,33,150,0.3);border:1px solid {BLUE_HEPATICA};border-radius:14px;padding:12px;text-align:center;'>
            <img src='data:image/png;base64,{b64_2}' style='max-width:270px;border-radius:8px'/>
        </div>
        <div style='background:rgba(82,33,150,0.3);border:1px solid {BLUE_HEPATICA};border-radius:14px;padding:12px;text-align:center;'>
            <img src='data:image/png;base64,{b64_3}' style='max-width:270px;border-radius:8px'/>
        </div>
    </div>
    """
    return HTML(get_css() + html)

# ─── WIDGETS ──────────────────────────────────────────────────────────────────
nama_input = widgets.Text(
    placeholder="Nama tugas", description="📝 Nama:",
    layout=widgets.Layout(width='500px'))
waktu_input = widgets.IntSlider(
    min=5, max=500, step=5, value=30, description="⏱ Menit:",
    layout=widgets.Layout(width='500px'))
deadline_input = widgets.DatePicker(
    description="📅 Deadline:", layout=widgets.Layout(width='500px'))
kategori_input = widgets.Dropdown(
    options=KATEGORI_LIST, value="Akademik",
    description="📂 Kategori:", layout=widgets.Layout(width='500px'))
difficulty_input = widgets.RadioButtons(
    options=[(f'{bintang(i)}  ({i})', i) for i in range(1, 6)],
    value=3, description="🎯 Kesulitan:",
    layout=widgets.Layout(width='500px'))

out_score_preview = widgets.Output()

def update_score_preview(*args):
    dl   = deadline_input.value.isoformat() if deadline_input.value else None
    skor, prio = hitung_skor(difficulty_input.value, dl)
    sc   = score_color(skor)
    pc   = {  "High": BANAISAGI_PURPLE,
              "Medium": BLUE_HEPATICA,
              "Low": BLUE_TOURMALIN}[prio]
    with out_score_preview:
        clear_output(wait=True)
        display(HTML(get_css() + f"""
        <div style='background:rgba(18,8,32,0.9);border:1px solid {BLUE_HEPATICA};border-radius:12px;
                    padding:12px 18px;display:inline-block;margin-top:6px;'>
            <span style='color:{FAIRY_LIGHTS};font-size:13px;opacity:0.7'>Estimasi Skor: </span>
            <span style='font-size:24px;font-weight:bold;color:{sc}'>{skor}</span>
            <span class='score-bar-wrap' style='width:120px;vertical-align:middle;margin:0 10px'>
                <span class='score-bar-fill' style='width:{skor}%;background:{sc}'></span>
            </span>
            <span style='color:{pc};font-weight:bold;font-size:14px'>→ {prio}</span>
        </div>"""))

difficulty_input.observe(update_score_preview, names='value')
deadline_input.observe(update_score_preview,   names='value')

btn_tambah = widgets.Button(description="➕ TAMBAH TUGAS", layout=widgets.Layout(width='300px'))
btn_reset  = widgets.Button(description="🗑 HAPUS SEMUA",  layout=widgets.Layout(width='300px'))
btn_tambah.add_class('cs-button')
btn_reset.add_class('cs-button')

out_tabel    = widgets.Output()
out_progress = widgets.Output()
out_jadwal   = widgets.Output()
out_status   = widgets.Output()
out_formula  = widgets.Output()
out_plot_kat = widgets.Output()

def refresh_all():
    recalc_all_scores()
    with out_tabel:
        clear_output(wait=True); display(HTML(get_css())); display(tampilkan_tabel())
    with out_progress:
        clear_output(wait=True); display(HTML(get_css())); display(tampil_progress_html())
    with out_jadwal:
        clear_output(wait=True); display(HTML(get_css())); display(buat_jadwal())
    with out_formula:
        clear_output(wait=True); display(tampil_formula())
    with out_plot_kat:
        clear_output(wait=True); display(HTML(get_css())); display(buat_plot_kategori())

def tambah(btn):
    with out_status:
        clear_output()
        nama = nama_input.value.strip()
        if not nama:
            print("⚠️ Nama tugas harus diisi!"); return
        dl = deadline_input.value.isoformat() if deadline_input.value else None
        add_task(nama, waktu_input.value, dl, kategori_input.value, difficulty_input.value)
        skor, prio = hitung_skor(difficulty_input.value, dl)
        print(f"✅ '{nama}' ditambahkan! Skor: {skor} → Prioritas: {prio}")
        nama_input.value = ""; waktu_input.value = 30
        deadline_input.value = None; kategori_input.value = "Akademik"
        difficulty_input.value = 3
        refresh_all(); update_score_preview()

def hapus_semua(btn):
    with out_status:
        clear_output()
        clear_all_tasks()
        print("🗑 Semua tugas telah dihapus.")
        refresh_all()
        insert_sample_tasks()
        refresh_all()

btn_tambah.on_click(tambah)
btn_reset.on_click(hapus_semua)

id_selesai  = widgets.IntText(description="ID Tugas:", value=1, layout=widgets.Layout(width='150px'))
btn_selesai = widgets.Button(description="✔️ TANDAI SELESAI", layout=widgets.Layout(width='180px'))
btn_selesai.add_class('cs-button')

def selesaikan(btn):
    with out_status:
        clear_output()
        tid = id_selesai.value
        sukses, pesan = complete_task_by_id(tid)
        if sukses:
            print(f"✅ Tugas ID {tid} telah diselesaikan!"); refresh_all()
        else:
            print(f"❌ Gagal: {pesan}. Cek ID {tid} dan pastikan status masih Pending.")
        id_selesai.value = 1

btn_selesai.on_click(selesaikan)

# ─── TAMPILAN UTAMA ───────────────────────────────────────────────────────────
display(HTML(get_css()))
display(HTML(f"""
<div class='cs-container'>
    <div class='cs-title'>ChronoScore</div>
    <div style='text-align:center'><span class='cs-sub'>TO DO LIST BY KELOMPOK 6</span></div>
"""))

display(HTML(f"<div class='cs-card'><h3>⚙️ SISTEM SCORING PRIORITAS</h3>"))
display(out_formula)
display(HTML("</div>"))

display(HTML(f"<div class='cs-card'><h3 style='text-align:center'>✍️ TAMBAH TUGAS BARU</h3>"
             "<div style='display:flex;justify-content:center;'>"))
display(widgets.VBox([
    nama_input, waktu_input, deadline_input, kategori_input,
    difficulty_input, out_score_preview,
    widgets.HBox([btn_tambah, btn_reset])
], layout=widgets.Layout(align_items='center')))
display(HTML("</div></div>"))

display(HTML(f"<div class='cs-card'><h3>📋 DAFTAR TUGAS (diurutkan by Skor)</h3>"))
display(out_tabel)
display(HTML("</div>"))

display(HTML(f"<div class='cs-card'><h3>✅ TANDAI TUGAS SELESAI</h3>"))
display(widgets.HBox([id_selesai, btn_selesai]))
display(HTML("</div>"))

display(HTML(f"<div class='cs-card'><h3>📊 PROGRESS & JADWAL</h3>"))
display(out_progress)
display(out_jadwal)
display(HTML("</div>"))

display(HTML(f"<div class='cs-card'><h3>📈 ANALISIS PER KATEGORI</h3>"))
display(out_plot_kat)
display(HTML("</div>"))

display(HTML("</div>"))
display(out_status)

refresh_all()
update_score_preview()

Output()

Output()

Output()

Output()

Output()

Output()